In [0]:
# Step 1: Read data from the table
df = spark.table("main.default.autoloader_csv_demo")

# Step 2: Write as Parquet to Volumes
parquet_path = "/Volumes/main/default/raw_data_volume/parquet_files/user_data/"
df.write.mode("overwrite").parquet(parquet_path)

print(f"Parquet files written to: {parquet_path}")

In [0]:
# Read Parquet files and create table
parquet_path = "/Volumes/main/default/raw_data_volume/parquet_files/user_data/"
df_parquet = spark.read.parquet(parquet_path)

# Save as Delta table
df_parquet.write.mode("overwrite").saveAsTable("main.default.user_data_external")

print(f"Table created: main.default.user_data_external")
print(f"Total rows: {df_parquet.count()}")

In [0]:
from delta.tables import DeltaTable

table_name = "main.default.user_data_external"
delta_table = DeltaTable.forName(spark, table_name)
spark.sql(f"OPTIMIZE {table_name} ZORDER BY (city, ingestion_time)")

In [0]:
# Without Z-ORDER: Scans ALL files
# With Z-ORDER: Scans ONLY relevant files (10-100x faster)

table_name = "main.default.user_data_external"
history_df = spark.sql(f"DESCRIBE HISTORY {table_name}")
detail_df = spark.sql(f"DESCRIBE DETAIL {table_name}")

In [0]:
# ============================================
# Complete Workflow: Time Travel, Restore, Vacuum
# ============================================
# How these three features work together

print("="*60)
print("DELTA LAKE DATA LIFECYCLE MANAGEMENT")
print("="*60)

print("\n=== Feature Comparison ===")
print("\n1. TIME TRAVEL (VERSION AS OF)")
print("   Purpose: Query historical data")
print("   Effect:  READ ONLY - doesn't change table")
print("   Syntax:  SELECT * FROM table VERSION AS OF 5")
print("   Use:     Audit, compare, debug\n")

print("2. RESTORE")
print("   Purpose: Rollback table to previous state")
print("   Effect:  WRITES - creates new version")
print("   Syntax:  RESTORE TABLE table TO VERSION AS OF 5")
print("   Use:     Undo mistakes, recover data\n")

print("3. VACUUM")
print("   Purpose: Delete old files to free storage")
print("   Effect:  PERMANENT deletion")
print("   Syntax:  VACUUM table RETAIN 168 HOURS")
print("   Use:     Save storage costs\n")

print("="*60)
print("COMPLETE WORKFLOW EXAMPLE")
print("="*60)

print("\nDay 1: Initial data load")
print("  INSERT INTO table VALUES (...) -- Version 0")
print("  Result: Table has 1000 rows\n")

print("Day 2: Add more data")
print("  INSERT INTO table VALUES (...) -- Version 1")
print("  Result: Table has 1500 rows\n")

print("Day 3: Update records")
print("  UPDATE table SET status = 'active' -- Version 2")
print("  Result: 500 rows updated\n")

print("Day 4: Accident! Wrong DELETE")
print("  DELETE FROM table WHERE city = 'Bangalore' -- Version 3")
print("  Result: Oops! Deleted 800 rows by mistake!\n")

print("Day 4: Use TIME TRAVEL to check before accident")
print("  SELECT COUNT(*) FROM table VERSION AS OF 2")
print("  Result: Shows 1500 rows (before DELETE)\n")

print("Day 4: Use RESTORE to fix mistake")
print("  RESTORE TABLE table TO VERSION AS OF 2")
print("  Result: Data restored! Creates Version 4\n")

print("Day 10: Clean up old files")
print("  VACUUM table RETAIN 168 HOURS  -- Keep 7 days")
print("  Result: Deleted files from Versions 0-3\n")

print("="*60)
print("RETENTION PERIOD IMPACT")
print("="*60)

print("\nBefore VACUUM (Day 10):")
print("  Version 0: Can time travel ✓")
print("  Version 1: Can time travel ✓")
print("  Version 2: Can time travel ✓")
print("  Version 3: Can time travel ✓")
print("  Version 4: Can time travel ✓\n")

print("After VACUUM RETAIN 168 HOURS (Day 10):")
print("  Version 0: DELETED (older than 7 days) ✗")
print("  Version 1: DELETED (older than 7 days) ✗")
print("  Version 2: DELETED (older than 7 days) ✗")
print("  Version 3: Available (within 7 days) ✓")
print("  Version 4: Available (within 7 days) ✓\n")

print("="*60)
print("BEST PRACTICES")
print("="*60)

print("\n✓ TIME TRAVEL:")
print("  • Use for auditing and debugging")
print("  • Query by VERSION or TIMESTAMP")
print("  • No cost - just reads existing files\n")

print("✓ RESTORE:")
print("  • Always check DESCRIBE HISTORY first")
print("  • Test with time travel before restoring")
print("  • RESTORE is fast (metadata operation)\n")

print("✓ VACUUM:")
print("  • Run DRY RUN first (mandatory!)")
print("  • Set retention based on business needs")
print("  • Schedule regularly (weekly/monthly)")
print("  • Balance storage cost vs time travel needs\n")

print("="*60)
print("QUICK REFERENCE COMMANDS")
print("="*60)

table_name = "main.default.user_data_external"

print(f"\n# View history")
print(f"DESCRIBE HISTORY {table_name};\n")

print(f"# Time travel to version 5")
print(f"SELECT * FROM {table_name} VERSION AS OF 5;\n")

print(f"# Time travel to timestamp")
print(f"SELECT * FROM {table_name} TIMESTAMP AS OF '2026-03-29';\n")

print(f"# Restore to version 5")
print(f"RESTORE TABLE {table_name} TO VERSION AS OF 5;\n")

print(f"# Vacuum dry run (safe)")
print(f"VACUUM {table_name} DRY RUN;\n")

print(f"# Vacuum with 7-day retention")
print(f"VACUUM {table_name} RETAIN 168 HOURS;\n")

print(f"# Check storage")
print(f"DESCRIBE DETAIL {table_name};")

print("\n" + "="*60)
print("END OF DELTA LAKE LIFECYCLE GUIDE")
print("="*60)

In [0]:
# ============================================
# VACUUM: Delete Old Data Files
# ============================================
# VACUUM removes old data files to free up storage
# These files accumulate from UPDATEs, DELETEs, MERGEs, and OPTIMIZE

from delta.tables import DeltaTable

table_name = "main.default.user_data_external"

print("=== What is VACUUM? ===")
print("VACUUM permanently deletes old data files that are no longer needed")
print("This frees up storage space\n")

print("=== Why do old files exist? ===")
print("Every operation creates new files:")
print("  • UPDATE/DELETE: Rewrites affected files")
print("  • MERGE: Creates new versions")
print("  • OPTIMIZE: Compacts small files")
print("Old files remain for time travel and RESTORE\n")

# Check current table size
print("=== Current Table Storage ===")
detail = spark.sql(f"DESCRIBE DETAIL {table_name}").collect()[0]
print(f"Number of files: {detail['numFiles']}")
print(f"Total size: {detail['sizeInBytes']:,} bytes ({detail['sizeInBytes'] / (1024*1024):.2f} MB)\n")

# Show how many versions exist
history = spark.sql(f"DESCRIBE HISTORY {table_name}")
version_count = history.count()
print(f"=== Version History ===")
print(f"Total versions: {version_count}")
print("Each version may have old files consuming storage\n")

print("=== VACUUM Syntax ===")
print("\nMethod 1: SQL - VACUUM with default retention (7 days)")
print(f"  VACUUM {table_name}")
print("\nMethod 2: SQL - VACUUM with custom retention")
print(f"  VACUUM {table_name} RETAIN 168 HOURS  -- 7 days")
print("\nMethod 3: Python API")
print(f"  delta_table = DeltaTable.forName(spark, '{table_name}')")
print(f"  delta_table.vacuum(168)  -- 168 hours = 7 days")
print("\nMethod 4: DRY RUN (see what will be deleted)")
print(f"  VACUUM {table_name} DRY RUN")

print("\n=== Default Retention Period ===")
print("⚠️ Default: 7 days (168 hours)")
print("⚠️ Files older than 7 days will be deleted")
print("⚠️ Cannot time travel beyond retention period after VACUUM\n")

print("=== Safety Features ===")
print("✓ VACUUM won't delete files from last 7 days (default)")
print("✓ DRY RUN shows files to be deleted without deleting")
print("✓ Retention period prevents accidental deletion\n")

print("=== When to VACUUM ===")
print("✓ After many UPDATEs/DELETEs (lots of old files)")
print("✓ After multiple OPTIMIZE operations")
print("✓ When storage costs are high")
print("✓ Regular schedule (weekly/monthly)\n")

print("=== VACUUM vs Time Travel ===")
print("Before VACUUM: Can time travel to ALL versions")
print("After VACUUM:  Can only time travel within retention period")
print("\nExample:")
print("  Retention = 7 days")
print("  Today: Can access versions from last 7 days")
print("  Version from 10 days ago: DELETED by VACUUM\n")

print("=== Important Warnings ===")
print("⚠️ VACUUM is PERMANENT - deleted files cannot be recovered")
print("⚠️ Always run DRY RUN first to see what will be deleted")
print("⚠️ Ensure no long-running queries are reading old files")
print("⚠️ Consider your time travel needs before vacuuming")

In [0]:
%sql
-- ============================================
-- VACUUM - SQL Examples
-- ============================================

-- Step 1: DRY RUN - See what will be deleted (safe)
VACUUM main.default.user_data_external DRY RUN;
-- Returns list of files that would be deleted
-- No actual deletion happens

-- Step 2: VACUUM with default retention (7 days)
-- Commented out - only run when you're sure
-- VACUUM main.default.user_data_external;

-- Step 3: VACUUM with custom retention period
-- Keep 30 days of history
-- VACUUM main.default.user_data_external RETAIN 720 HOURS;  -- 30 days

-- Keep only 24 hours (risky!)
-- VACUUM main.default.user_data_external RETAIN 24 HOURS;

-- Step 4: Check storage after VACUUM
-- DESCRIBE DETAIL main.default.user_data_external;
-- numFiles should be lower
-- sizeInBytes should be smaller

-- ============================================
-- VACUUM Best Practices
-- ============================================

-- Practice 1: Always DRY RUN first
-- VACUUM table DRY RUN;

-- Practice 2: Set appropriate retention based on needs
-- Compliance requirement: Keep 90 days
-- VACUUM table RETAIN 2160 HOURS;  -- 90 days

-- Practice 3: Schedule VACUUM regularly
-- Run weekly or monthly via Databricks Jobs

-- Practice 4: Monitor storage savings
-- Compare DESCRIBE DETAIL before and after VACUUM

In [0]:
# ============================================
# RESTORE: Rollback to Previous Version
# ============================================
# RESTORE reverts your table to a previous state
# Think of it as UNDO for your entire table

from delta.tables import DeltaTable

table_name = "main.default.user_data_external"

print("=== What is RESTORE? ===")
print("RESTORE rolls back table to a previous version")
print("Use cases:")
print("  • Undo accidental DELETE/UPDATE")
print("  • Revert after bad data load")
print("  • Recover from mistakes\n")

# Check current state
print("=== Current Table State ===")
current_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {table_name}").collect()[0]['cnt']
print(f"Current row count: {current_count}")

# Show history to pick a version
history = spark.sql(f"DESCRIBE HISTORY {table_name}")
latest_version = history.agg({"version": "max"}).collect()[0][0]

print(f"\nCurrent version: {latest_version}")
print("\nVersion history:")
for row in history.select("version", "timestamp", "operation").limit(5).collect():
    print(f"  v{row['version']}: {row['operation']} at {row['timestamp']}")

# IMPORTANT: This is a demonstration - don't actually restore unless needed
print("\n=== RESTORE Syntax ===")
print("\nMethod 1: Restore to VERSION")
print(f"  RESTORE TABLE {table_name} TO VERSION AS OF 0")
print("\nMethod 2: Restore to TIMESTAMP")
print(f"  RESTORE TABLE {table_name} TO TIMESTAMP AS OF '2026-03-29T00:00:00'")
print("\nMethod 3: Python API")
print(f"  DeltaTable.forName(spark, '{table_name}').restoreToVersion(0)")

# Example scenario (commented out to avoid accidental execution)
print("\n=== Example Scenario ===")
print("# Step 1: Accident happens")
print("# DELETE FROM table WHERE city = 'Bangalore'  -- Oops! Wrong WHERE clause!\n")

print("# Step 2: Check history to find version before mistake")
print("# DESCRIBE HISTORY table\n")

print("# Step 3: Restore to version before the accident")
print("# RESTORE TABLE table TO VERSION AS OF 3  -- Go back to version 3\n")

print("# Step 4: Verify data is back")
print("# SELECT COUNT(*) FROM table  -- Data restored!\n")

print("=== Important Notes ===")
print("⚠️ RESTORE creates a NEW version (doesn't delete history)")
print("⚠️ Old versions still exist and use storage")
print("⚠️ Use VACUUM to clean up old files (see next cell)")
print("✓ RESTORE is metadata-only operation (fast!)")
print("✓ Data files are reused, not copied")

In [0]:
%sql
-- ============================================
-- RESTORE TABLE - SQL Examples
-- ============================================

-- Show current table history
DESCRIBE HISTORY main.default.user_data_external;

-- RESTORE by VERSION (commented out - only run if you need to rollback)
-- RESTORE TABLE main.default.user_data_external TO VERSION AS OF 0;

-- RESTORE by TIMESTAMP (commented out)
-- RESTORE TABLE main.default.user_data_external 
-- TO TIMESTAMP AS OF '2026-03-29T00:00:00';

-- After RESTORE, check the table
-- SELECT * FROM main.default.user_data_external LIMIT 10;

-- RESTORE creates a new version
-- DESCRIBE HISTORY main.default.user_data_external;  -- You'll see a RESTORE operation

In [0]:
# ============================================
# STEP 5: DIAMOND ORCHESTRATION
# ============================================
# Pattern: Task A ─┬→ Task B ─┐
#                 └→ Task C ─┤→ Task D
# Parallel processing then combine

print("="*80)
print("DIAMOND ORCHESTRATION - PARALLEL + MERGE")
print("="*80)

print("""
Use Case: ML Feature Engineering
  Task 1: Load raw data
  Task 2: Compute user demographics (parallel)
  Task 3: Compute behavioral features (parallel)
  Task 4: Combine features and train model

Execution Flow:
                 ─┬→ [Demographics] ─┐
  [Load Data] ──┤                      ├→ [Train Model]
                 └→ [Behavioral]   ─┘
  
Diamond shape: Split (fan-out) then merge (fan-in)
""")

print("\n" + "="*80)
print("CODE: Create Diamond Pattern Job")
print("="*80)

print("""
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient()

job = w.jobs.create(
    name="ML Feature Engineering Pipeline",
    tasks=[
        # === INITIAL TASK ===
        
        # Task 1: Load raw data
        jobs.Task(
            task_key="load_raw_data",
            description="Load and prepare raw dataset",
            notebook_task=jobs.NotebookTask(
                notebook_path="/ML/01_load_data",
                base_parameters={"date": "{{job.start_time.iso_date}}"}
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=3
            )
        ),
        
        # === PARALLEL FEATURE ENGINEERING (Fan-Out) ===
        
        # Task 2: Demographics features
        jobs.Task(
            task_key="compute_demographics",
            description="Compute demographic features",
            depends_on=[jobs.TaskDependency(task_key="load_raw_data")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/ML/02_demographics_features"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=2
            )
        ),
        
        # Task 3: Behavioral features (parallel with demographics)
        jobs.Task(
            task_key="compute_behavioral",
            description="Compute behavioral features",
            depends_on=[jobs.TaskDependency(task_key="load_raw_data")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/ML/02_behavioral_features"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=3
            )
        ),
        
        # Task 4: Transaction features (parallel with above)
        jobs.Task(
            task_key="compute_transactions",
            description="Compute transaction features",
            depends_on=[jobs.TaskDependency(task_key="load_raw_data")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/ML/02_transaction_features"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=2
            )
        ),
        
        # === MERGE AND TRAIN (Fan-In) ===
        
        # Task 5: Combine features and train
        jobs.Task(
            task_key="train_model",
            description="Combine all features and train ML model",
            depends_on=[
                jobs.TaskDependency(task_key="compute_demographics"),
                jobs.TaskDependency(task_key="compute_behavioral"),
                jobs.TaskDependency(task_key="compute_transactions")
            ],
            notebook_task=jobs.NotebookTask(
                notebook_path="/ML/03_train_model"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="g4dn.xlarge",  # GPU for ML
                num_workers=0  # Single-node
            ),
            libraries=[
                jobs.Library(pypi=jobs.PythonPyPiLibrary(package="xgboost==2.0.0")),
                jobs.Library(pypi=jobs.PythonPyPiLibrary(package="mlflow==2.10.0"))
            ]
        ),
        
        # === EVALUATION ===
        
        # Task 6: Evaluate and log
        jobs.Task(
            task_key="evaluate_model",
            description="Evaluate model performance",
            depends_on=[jobs.TaskDependency(task_key="train_model")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/ML/04_evaluate"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=1
            )
        )
    ],
    
    schedule=jobs.CronSchedule(
        quartz_cron_expression="0 0 3 * * ? *",  # 3 AM daily
        timezone_id="UTC"
    )
)

print(f"Diamond pattern job created: {job.job_id}")
""")

print("\n" + "="*80)
print("EXECUTION FLOW")
print("="*80)

print("""
When job runs:

3:00 AM - Job starts
3:00 AM - [Load Raw Data] starts
3:10 AM - [Load Raw Data] completes (10 min)

        ┌────── PARALLEL (Fan-Out) ──────┐
3:10 AM - [Demographics] starts
3:10 AM - [Behavioral] starts
3:10 AM - [Transactions] starts
        └────────────────────────────────┘

3:15 AM - [Demographics] completes (5 min)
3:20 AM - [Transactions] completes (10 min)
3:25 AM - [Behavioral] completes (15 min) ← Longest

        ┌───── MERGE (Fan-In) ─────┐
3:25 AM - [Train Model] starts (all features ready)
        └────────────────────────────┘

3:55 AM - [Train Model] completes (30 min)
3:55 AM - [Evaluate Model] starts
4:00 AM - [Evaluate Model] completes (5 min)
4:00 AM - Job completes ✓

Total time: 60 minutes

Breakdown:
  Load: 10 min
  Parallel features: 15 min (not 5+10+15=30!)
  Train: 30 min
  Evaluate: 5 min
  Total: 60 min (vs 90 min sequential)
""")

print("\n" + "="*80)
print("VISUAL REPRESENTATION")
print("="*80)

print("""
        [Load Raw Data] (10 min)
               │
      ┌──────┼──────┐
      │       │       │
  [Demographics] │  [Transactions]
    (5 min)      │     (10 min)
      │    [Behavioral]  │
      │     (15 min)     │
      │          │       │
      └────────┼───────┘
                 │
          [Train Model] (30 min)
                 │
         [Evaluate Model] (5 min)

Diamond shape! ◇
""")

print("\n" + "="*80)
print("KEY OBSERVATIONS")
print("="*80)

print("""
1. SPLIT POINT
   • After loading, work splits into 3 parallel tasks
   • Each computes different feature set
   • Run simultaneously on separate clusters

2. PARALLEL EXECUTION
   • 3 feature tasks run for 15 minutes total
   • Would be 30 min sequential (5+10+15)
   • 50% time savings in this phase

3. MERGE POINT
   • Training waits for ALL features
   • Combines all feature sets
   • Single bottleneck

4. BALANCED DESIGN
   • Parallel phase saves time
   • Merge phase does complex work
   • Overall efficient
""")

print("\n" + "="*80)
print("PROS AND CONS")
print("="*80)

print("""
✓ ADVANTAGES:
  • Best of both worlds (parallel + coordination)
  • Natural for multi-step transformations
  • Common in ML pipelines
  • Efficient resource usage
  • Clear separation of concerns

✗ DISADVANTAGES:
  • More complex than simple patterns
  • Requires careful design
  • Merge task may be bottleneck
  • Need to balance parallel task times
""")

print("\n" + "="*80)
print("WHEN TO USE DIAMOND")
print("="*80)

print("""
✓ Use Diamond when:
  • ML feature engineering from multiple sources
  • Multi-stage data transformations
  • Independent computations need merging
  • Balance parallelism and synchronization
  • Common base data, different processing

✗ Don't use when:
  • Simple linear pipeline (use sequential)
  • Tasks don't share base data
  • No natural split/merge points
  • Adds unnecessary complexity
""")

print("\n" + "="*80)
print("OPTIMIZATION TIP: Balance Parallel Tasks")
print("="*80)

print("""
Problem: Unbalanced parallel tasks
  Task A: 5 min   ─┐
  Task B: 30 min  ─┤→ Merge waits 30 min!
  Task C: 10 min  ─┘

Total time: 30 min (bottlenecked by Task B)

Solution: Break down long tasks
  Task A: 5 min   ─┐
  Task B1: 10 min ─┤→ Merge 1
  Task B2: 10 min ─┤→ Merge 2
  Task B3: 10 min ─┤→ Final Merge
  Task C: 10 min  ─┘

Or: Optimize slow task (more workers, better code)

Goal: Keep parallel tasks roughly same duration
""")

print("\n" + "="*80)
print("FEATURE ENGINEERING EXAMPLE")
print("="*80)

print("""
# Task 1: Load Raw Data
df_users = spark.table("users")
df_transactions = spark.table("transactions")
df_clickstream = spark.table("clickstream")

# Save for parallel tasks
df_users.write.mode("overwrite").saveAsTable("temp.users_raw")
df_transactions.write.mode("overwrite").saveAsTable("temp.transactions_raw")
df_clickstream.write.mode("overwrite").saveAsTable("temp.clickstream_raw")


# Task 2: Demographics Features (Parallel)
users = spark.table("temp.users_raw")
demographics = users.select(
    "user_id",
    "age",
    "gender",
    "city",
    "account_age_days"
)
demographics.write.mode("overwrite").saveAsTable("temp.demographics_features")


# Task 3: Behavioral Features (Parallel)
clicks = spark.table("temp.clickstream_raw")
behavioral = clicks.groupBy("user_id").agg(
    count("*").alias("total_clicks"),
    countDistinct("page").alias("unique_pages"),
    avg("session_duration").alias("avg_session_duration")
)
behavioral.write.mode("overwrite").saveAsTable("temp.behavioral_features")


# Task 4: Transaction Features (Parallel)
txns = spark.table("temp.transactions_raw")
transactions = txns.groupBy("user_id").agg(
    sum("amount").alias("total_spent"),
    count("*").alias("transaction_count"),
    avg("amount").alias("avg_transaction_value")
)
transactions.write.mode("overwrite").saveAsTable("temp.transaction_features")


# Task 5: Combine and Train (Merge)
demographics = spark.table("temp.demographics_features")
behavioral = spark.table("temp.behavioral_features")
transactions = spark.table("temp.transaction_features")

# Join all features
features = (demographics
    .join(behavioral, "user_id", "left")
    .join(transactions, "user_id", "left")
    .fillna(0)
)

# Train model
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run():
    X = features.drop("user_id", "target")
    y = features.select("target")
    
    model = RandomForestClassifier()
    model.fit(X, y)
    
    mlflow.sklearn.log_model(model, "model")
""")

print("\n" + "="*80)
print("END OF DIAMOND ORCHESTRATION")
print("="*80)

In [0]:
# ============================================
# REAL-WORLD ORCHESTRATION EXAMPLE
# ============================================
# Complete e-commerce data pipeline using ALL patterns

print("="*80)
print("COMPLETE ORCHESTRATION: E-COMMERCE ANALYTICS PIPELINE")
print("="*80)

print("""
Scenario: Daily end-to-end data pipeline for e-commerce platform

Business Requirements:
  1. Ingest data from multiple sources
  2. Clean and validate data
  3. Run data quality checks
  4. If quality passes: Update production
  5. If quality fails: Quarantine and alert
  6. Train recommendation model
  7. Update dashboards
  8. Send daily report

Orchestration Patterns Used:
  ✓ Sequential: For dependent tasks
  ✓ Parallel (Fan-Out): Multi-source ingestion
  ✓ Fan-In (Merge): Consolidate data
  ✓ Conditional (If-Then): Quality gates
  ✓ Diamond: ML feature engineering
""")

print("\n" + "="*80)
print("PIPELINE ARCHITECTURE")
print("="*80)

print("""
                    [Initialize]
                         │
      ┌────────────┼────────────┐
      │             │              │
  [Ingest S3]  [Ingest DB]  [Ingest API]  ← PARALLEL
      │             │              │
      └────────────┼────────────┘
                     │
               [Merge Data]                ← FAN-IN
                     │
              [Clean & Transform]
                     │
              [Quality Check]              ← DECISION POINT
                     │
      ┌────────────┼────────────┐
      │             │              │
   [Pass]         [Fail]            │    ← CONDITIONAL
      │             │              │
      │       [Quarantine]         │
      │             │              │
      │       [Send Alert]         │
      │                            │
      └────────────────────────────┘
                     │
           [Promote to Production]
                     │
      ┌────────────┼────────────┐
      │             │              │
 [User Feat]  [Product Feat] [Txn Feat]  ← PARALLEL (Diamond)
      │             │              │
      └────────────┼────────────┘
                     │
            [Train Model]                ← FAN-IN (Diamond)
                     │
            [Evaluate Model]
                     │
           [Update Dashboards]
                     │
            [Send Report]
                     │
                   [Done]
""")

print("\n" + "="*80)
print("COMPLETE CODE")
print("="*80)

print("""
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient()

job = w.jobs.create(
    name="E-commerce Daily Analytics Pipeline",
    tasks=[
        # ========================================
        # PHASE 1: INITIALIZATION
        # ========================================
        
        jobs.Task(
            task_key="initialize",
            description="Initialize pipeline, set parameters",
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipeline/00_initialize",
                base_parameters={
                    "run_date": "{{job.start_time.iso_date}}",
                    "environment": "production"
                }
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=1
            )
        ),
        
        # ========================================
        # PHASE 2: PARALLEL INGESTION (Fan-Out)
        # ========================================
        
        jobs.Task(
            task_key="ingest_s3",
            description="Ingest order data from S3",
            depends_on=[jobs.TaskDependency(task_key="initialize")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipeline/01_ingest_s3"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=4
            )
        ),
        
        jobs.Task(
            task_key="ingest_database",
            description="Ingest customer data from PostgreSQL",
            depends_on=[jobs.TaskDependency(task_key="initialize")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipeline/01_ingest_database"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=2
            )
        ),
        
        jobs.Task(
            task_key="ingest_api",
            description="Ingest product data from API",
            depends_on=[jobs.TaskDependency(task_key="initialize")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipeline/01_ingest_api"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=2
            )
        ),
        
        # ========================================
        # PHASE 3: MERGE DATA (Fan-In)
        # ========================================
        
        jobs.Task(
            task_key="merge_data",
            description="Merge all ingested data sources",
            depends_on=[
                jobs.TaskDependency(task_key="ingest_s3"),
                jobs.TaskDependency(task_key="ingest_database"),
                jobs.TaskDependency(task_key="ingest_api")
            ],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipeline/02_merge_data"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.2xlarge",
                num_workers=6
            )
        ),
        
        # ========================================
        # PHASE 4: TRANSFORMATION (Sequential)
        # ========================================
        
        jobs.Task(
            task_key="clean_transform",
            description="Clean and transform data",
            depends_on=[jobs.TaskDependency(task_key="merge_data")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipeline/03_clean_transform"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=4
            )
        ),
        
        # ========================================
        # PHASE 5: QUALITY CHECK (Decision Point)
        # ========================================
        
        jobs.Task(
            task_key="quality_check",
            description="Run data quality validation",
            depends_on=[jobs.TaskDependency(task_key="clean_transform")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipeline/04_quality_check"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=2
            )
        ),
        
        # === SUCCESS PATH ===
        
        jobs.Task(
            task_key="promote_production",
            description="Promote data to production tables",
            depends_on=[
                jobs.TaskDependency(
                    task_key="quality_check",
                    outcome="true"  # Only if quality passed
                )
            ],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipeline/05_promote_production"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=3
            )
        ),
        
        # === FAILURE PATH ===
        
        jobs.Task(
            task_key="quarantine_data",
            description="Move failed data to quarantine",
            depends_on=[
                jobs.TaskDependency(
                    task_key="quality_check",
                    outcome="false"  # Only if quality failed
                )
            ],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipeline/05_quarantine"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=1
            )
        ),
        
        jobs.Task(
            task_key="send_alert",
            description="Alert team about quality issues",
            depends_on=[jobs.TaskDependency(task_key="quarantine_data")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipeline/06_send_alert"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=1
            )
        ),
        
        # ========================================
        # PHASE 6: ML FEATURES (Diamond Pattern)
        # ========================================
        
        jobs.Task(
            task_key="compute_user_features",
            description="Compute user behavioral features",
            depends_on=[jobs.TaskDependency(task_key="promote_production")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/ML/01_user_features"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=3
            )
        ),
        
        jobs.Task(
            task_key="compute_product_features",
            description="Compute product affinity features",
            depends_on=[jobs.TaskDependency(task_key="promote_production")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/ML/01_product_features"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=2
            )
        ),
        
        jobs.Task(
            task_key="compute_transaction_features",
            description="Compute transaction pattern features",
            depends_on=[jobs.TaskDependency(task_key="promote_production")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/ML/01_transaction_features"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=2
            )
        ),
        
        # ========================================
        # PHASE 7: ML TRAINING (Merge)
        # ========================================
        
        jobs.Task(
            task_key="train_recommendation_model",
            description="Train product recommendation model",
            depends_on=[
                jobs.TaskDependency(task_key="compute_user_features"),
                jobs.TaskDependency(task_key="compute_product_features"),
                jobs.TaskDependency(task_key="compute_transaction_features")
            ],
            notebook_task=jobs.NotebookTask(
                notebook_path="/ML/02_train_model"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="g4dn.xlarge",
                num_workers=0
            ),
            libraries=[
                jobs.Library(pypi=jobs.PythonPyPiLibrary(package="xgboost==2.0.0")),
                jobs.Library(pypi=jobs.PythonPyPiLibrary(package="mlflow==2.10.0"))
            ]
        ),
        
        jobs.Task(
            task_key="evaluate_model",
            description="Evaluate model performance",
            depends_on=[jobs.TaskDependency(task_key="train_recommendation_model")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/ML/03_evaluate"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=1
            )
        ),
        
        # ========================================
        # PHASE 8: REPORTING (Sequential)
        # ========================================
        
        jobs.Task(
            task_key="update_dashboards",
            description="Refresh BI dashboards",
            depends_on=[jobs.TaskDependency(task_key="evaluate_model")],
            sql_task=jobs.SqlTask(
                warehouse_id="<sql-warehouse-id>",
                query=jobs.SqlTaskQuery(
                    query_id="<query-id>"
                )
            )
        ),
        
        jobs.Task(
            task_key="send_daily_report",
            description="Generate and email daily report",
            depends_on=[jobs.TaskDependency(task_key="update_dashboards")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Reporting/daily_report"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=1
            )
        )
    ],
    
    # Schedule: Daily at 2 AM
    schedule=jobs.CronSchedule(
        quartz_cron_expression="0 0 2 * * ? *",
        timezone_id="America/Los_Angeles"
    ),
    
    # Notifications
    email_notifications=jobs.JobEmailNotifications(
        on_success=["data-team@company.com"],
        on_failure=["oncall@company.com", "data-lead@company.com"]
    ),
    
    # Job settings
    max_concurrent_runs=1,
    timeout_seconds=14400  # 4 hours
)

print(f"Complete pipeline created: {job.job_id}")
""")

print("\n" + "="*80)
print("EXECUTION TIMELINE")
print("="*80)

print("""
2:00 AM - [Initialize] (2 min)

2:02 AM - [Ingest S3], [Ingest DB], [Ingest API] start (parallel)
2:17 AM - All ingestion completes (15 min - longest task)

2:17 AM - [Merge Data] (20 min)
2:37 AM - [Clean & Transform] (15 min)
2:52 AM - [Quality Check] (5 min)

         DECISION: Quality Passed ✓

2:57 AM - [Promote to Production] (10 min)

3:07 AM - [User Feat], [Product Feat], [Txn Feat] start (parallel)
3:22 AM - All features complete (15 min - longest)

3:22 AM - [Train Model] (30 min)
3:52 AM - [Evaluate Model] (5 min)
3:57 AM - [Update Dashboards] (2 min)
3:59 AM - [Send Report] (2 min)

4:01 AM - Pipeline complete ✓

Total: 2 hours 1 minute

Tasks executed: 16 total
Tasks skipped: 2 (quarantine path)
""")

print("\n" + "="*80)
print("KEY INSIGHTS")
print("="*80)

print("""
1. PATTERN DIVERSITY
   ✓ Used all 5 orchestration patterns
   ✓ Each pattern serves specific purpose
   ✓ Combined seamlessly

2. EFFICIENCY
   ✓ Parallel ingestion saves 30 min
   ✓ Parallel features save 20 min
   ✓ Total: ~50 min saved vs sequential

3. RELIABILITY
   ✓ Quality gates prevent bad data
   ✓ Conditional paths handle failures
   ✓ Automatic alerts on issues

4. MAINTAINABILITY
   ✓ Clear phases and dependencies
   ✓ Easy to debug and modify
   ✓ Well-structured pipeline
""")

print("\n" + "="*80)
print("MONITORING AND OBSERVABILITY")
print("="*80)

print("""
Databricks Jobs UI provides:
  ✓ Visual DAG of all tasks
  ✓ Real-time execution status
  ✓ Task-level logs and metrics
  ✓ Historical run data
  ✓ Duration trends
  ✓ Cost per run

Alerts configured for:
  ✓ Job failures
  ✓ Quality check failures
  ✓ Long-running tasks
  ✓ Cost thresholds
""")

print("\n" + "="*80)
print("END OF REAL-WORLD ORCHESTRATION EXAMPLE")
print("="*80)
print("\nYou now have a complete understanding of task orchestration!")
print("Use these patterns to build robust, efficient data pipelines.")
print("="*80)

In [0]:
# ============================================
# TASK ORCHESTRATION - SUMMARY & BEST PRACTICES
# ============================================

print("="*80)
print("ORCHESTRATION PATTERNS - COMPLETE SUMMARY")
print("="*80)

print("""
┌────────────────────────────────────────────────────────────────────────────┐
│ PATTERN 1: SEQUENTIAL (Linear)                                            │
│ ───────────────────────────────────────────────────────────────────────── │
│ Flow:      Task A → Task B → Task C                                      │
│ Execution: One at a time, in order                                       │
│ Duration:  Sum of all task times                                         │
│ Use When:  Tasks MUST run in specific order                              │
│ Examples:  ETL pipelines, data validation workflows                      │
└────────────────────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────────────────────┐
│ PATTERN 2: PARALLEL / FAN-OUT (Broadcast)                                 │
│ ───────────────────────────────────────────────────────────────────────── │
│ Flow:      Task A ─┬→ Task B                                             │
│                   ├→ Task C                                             │
│                   └→ Task D                                             │
│ Execution: B, C, D run simultaneously                                    │
│ Duration:  Max(B, C, D) time                                             │
│ Use When:  Independent tasks, no dependencies                            │
│ Examples:  Multi-source ingestion, parallel reports                      │
└────────────────────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────────────────────┐
│ PATTERN 3: FAN-IN / MERGE (Synchronization)                              │
│ ───────────────────────────────────────────────────────────────────────── │
│ Flow:      Task A ─┐                                                    │
│            Task B ─┤→ Task D                                            │
│            Task C ─┘                                                    │
│ Execution: D waits for A, B, C to complete                               │
│ Duration:  Max(A,B,C) + D time                                           │
│ Use When:  Need to combine/join multiple datasets                        │
│ Examples:  Feature joins for ML, data aggregation                        │
└────────────────────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────────────────────┐
│ PATTERN 4: CONDITIONAL (Decision Tree)                                    │
│ ───────────────────────────────────────────────────────────────────────── │
│ Flow:      Task A → if success → Task B                                  │
│                   │                                                      │
│                   └→ if failure → Task C                                  │
│ Execution: Dynamic path based on condition                               │
│ Duration:  Depends on path taken                                         │
│ Use When:  Data quality gates, error handling                            │
│ Examples:  Quality checks, A/B testing, retries                          │
└────────────────────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────────────────────┐
│ PATTERN 5: DIAMOND (Parallel + Merge)                                     │
│ ───────────────────────────────────────────────────────────────────────── │
│ Flow:                ─┬→ Task B ─┐                                     │
│            Task A ──┤           ├→ Task D                             │
│                     └→ Task C ─┘                                     │
│ Execution: Split, parallel process, then merge                           │
│ Duration:  A + Max(B,C) + D time                                         │
│ Use When:  Multi-step transformations                                    │
│ Examples:  ML pipelines, complex ETL                                     │
└────────────────────────────────────────────────────────────────────────────┘
""")

print("\n" + "="*80)
print("DECISION GUIDE - WHICH PATTERN TO USE?")
print("="*80)

print("""
Ask yourself these questions:

1. Do tasks DEPEND on each other?
   YES → Sequential
   NO  → Go to #2

2. Can tasks run SIMULTANEOUSLY?
   YES → Go to #3
   NO  → Sequential

3. Do you need to COMBINE results later?
   YES → Diamond (if starting from one task) or Fan-In (if multiple starts)
   NO  → Parallel (Fan-Out)

4. Does the workflow have CONDITIONAL logic?
   YES → Conditional (with any other pattern)
   NO  → Use patterns from above

5. Is this a COMPLEX, multi-phase workflow?
   YES → Combine multiple patterns
   NO  → Stick with one simple pattern
""")

print("\n" + "="*80)
print("BEST PRACTICES")
print("="*80)

print("""
✓ START SIMPLE
  • Begin with Sequential for MVP
  • Add parallelism where bottlenecks exist
  • Iterate based on performance metrics

✓ BALANCE PARALLELISM
  • Don't over-parallelize (resource contention)
  • Keep parallel tasks similar duration
  • Monitor cluster utilization

✓ HANDLE FAILURES GRACEFULLY
  • Use Conditional patterns for error paths
  • Set retry policies (max_retries, timeout)
  • Add notifications for critical failures
  • Implement dead-letter queues for bad data

✓ OPTIMIZE FOR COST
  • Use job clusters (cheaper than all-purpose)
  • Enable autoscaling
  • Right-size cluster for each task
  • Terminate clusters when idle

✓ MONITOR & ALERT
  • Track execution time trends
  • Set SLAs and alert on violations
  • Log task dependencies clearly
  • Use Databricks Job UI for visualization

✓ VERSION CONTROL
  • Store job definitions in Git
  • Use CI/CD for deployment
  • Test pipelines in dev/staging first
  • Document complex workflows
""")

print("\n" + "="*80)
print("COMMON ANTI-PATTERNS (AVOID!)")
print("="*80)

print("""
✗ MONOLITHIC JOBS
  Problem: One giant notebook doing everything
  Solution: Break into logical, testable tasks

✗ OVER-PARALLELIZATION
  Problem: 50 tasks running simultaneously
  Solution: Group related work, limit concurrency

✗ NO ERROR HANDLING
  Problem: Pipeline fails silently
  Solution: Add retries, alerts, conditional paths

✗ HARD-CODED VALUES
  Problem: Cannot reuse across environments
  Solution: Use parameters, environment variables

✗ TIGHT COUPLING
  Problem: Tasks directly reference each other's internals
  Solution: Use well-defined interfaces (tables, files)

✗ NO MONITORING
  Problem: Don't know when/why jobs fail
  Solution: Set up alerts, dashboards, logging
""")

print("\n" + "="*80)
print("PATTERN COMPLEXITY COMPARISON")
print("="*80)

print("""
┌────────────────┬────────────┬────────────┬──────────────┐
│ Pattern         │ Complexity │ Speedup    │ Maintenance  │
├────────────────┼────────────┼────────────┼──────────────┤
│ Sequential      │ Low        │ 1x (base)  │ Easy         │
│ Parallel        │ Medium     │ 2-5x       │ Medium       │
│ Fan-In          │ Medium     │ 2-4x       │ Medium       │
│ Conditional     │ High       │ Varies     │ Complex      │
│ Diamond         │ High       │ 3-8x       │ Complex      │
└────────────────┴────────────┴────────────┴──────────────┘
""")

print("\n" + "="*80)
print("QUICK REFERENCE - CODE TEMPLATES")
print("="*80)

print("""
# SEQUENTIAL
jobs.Task(
    task_key="task_1",
    notebook_task=...
),
jobs.Task(
    task_key="task_2",
    depends_on=[jobs.TaskDependency(task_key="task_1")],
    notebook_task=...
)

# PARALLEL
jobs.Task(task_key="task_1", ...),  # Trigger
jobs.Task(
    task_key="task_2",
    depends_on=[jobs.TaskDependency(task_key="task_1")],
    ...
),
jobs.Task(
    task_key="task_3",
    depends_on=[jobs.TaskDependency(task_key="task_1")],
    ...
)

# FAN-IN
jobs.Task(task_key="task_1", ...),
jobs.Task(task_key="task_2", ...),
jobs.Task(
    task_key="merge",
    depends_on=[
        jobs.TaskDependency(task_key="task_1"),
        jobs.TaskDependency(task_key="task_2")
    ],
    ...
)

# CONDITIONAL
jobs.Task(
    task_key="check",
    condition_task=jobs.ConditionTask(
        left="{{tasks.quality_check.values.row_count}}",
        op=jobs.ConditionTaskOp.GREATER_THAN,
        right="1000"
    ),
    depends_on=[jobs.TaskDependency(task_key="quality_check")]
)
""")

print("\n" + "="*80)
print("LEARNING PATH RECOMMENDATION")
print("="*80)

print("""
Week 1: Master Sequential
  • Build 2-3 simple ETL pipelines
  • Practice notebook tasks, SQL tasks
  • Get comfortable with scheduling

Week 2: Add Parallelism
  • Convert sequential jobs to parallel
  • Measure performance improvements
  • Learn resource optimization

Week 3: Implement Merging
  • Build fan-in patterns
  • Practice data joining/aggregation
  • Handle timing issues

Week 4: Add Intelligence
  • Implement conditional logic
  • Add error handling paths
  • Build complete diamond workflows

Week 5: Production Ready
  • Add monitoring and alerts
  • Implement retry logic
  • Document and version control
  • Deploy to production
""")

print("\n" + "="*80)
print("RESOURCES & NEXT STEPS")
print("="*80)

print("""
✓ Databricks Documentation:
  • Jobs API: docs.databricks.com/workflows/jobs/jobs-api.html
  • Task dependencies: docs.databricks.com/workflows/jobs/how-to/

✓ Best Practices:
  • Workflow orchestration guide
  • Performance optimization
  • Cost management

✓ Advanced Topics:
  • Delta Live Tables (declarative pipelines)
  • Databricks Workflows (multi-task orchestration)
  • Apache Airflow integration
  • Event-driven orchestration

✓ Try It Yourself:
  1. Pick one pattern from this notebook
  2. Adapt code to your use case
  3. Create and run the job
  4. Monitor performance
  5. Iterate and improve
""")

print("\n" + "="*80)
print("END OF TASK ORCHESTRATION TUTORIAL")
print("="*80)
print("\nYou now have all 5 orchestration patterns!")
print("Start with Sequential, then gradually add complexity.")
print("\n🚀 Happy orchestrating! 🚀\n")

In [0]:
# ============================================
# STEP 3: FAN-IN ORCHESTRATION (MERGE)
# ============================================
# Pattern: Task A ─┐
#          Task B ─┤→ Task D
#          Task C ─┘
# Wait for multiple tasks before proceeding

print("="*80)
print("FAN-IN ORCHESTRATION - MERGE PATTERN")
print("="*80)

print("""
Use Case: Data Consolidation Pipeline
  Task 1: Ingest customers (parallel)
  Task 2: Ingest orders (parallel)
  Task 3: Ingest products (parallel)
  Task 4: Join all data (waits for 1, 2, 3)
  Task 5: Create analytics tables

Execution Flow:
  [Customers] ─┐
  [Orders]    ─┤→ [Join All] → [Analytics]
  [Products]  ─┘
  
Task 4 waits for ALL upstream tasks to complete
""")

print("\n" + "="*80)
print("CODE: Create Fan-In Job")
print("="*80)

print("""
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient()

job = w.jobs.create(
    name="Fan-In Data Consolidation",
    tasks=[
        # === PARALLEL INGESTION (Fan-Out) ===
        
        # Task 1: Ingest customers
        jobs.Task(
            task_key="ingest_customers",
            description="Ingest customer data",
            notebook_task=jobs.NotebookTask(
                notebook_path="/Ingestion/customers"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=2
            )
        ),
        
        # Task 2: Ingest orders (parallel with Task 1)
        jobs.Task(
            task_key="ingest_orders",
            description="Ingest order data",
            notebook_task=jobs.NotebookTask(
                notebook_path="/Ingestion/orders"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=3
            )
        ),
        
        # Task 3: Ingest products (parallel with Tasks 1, 2)
        jobs.Task(
            task_key="ingest_products",
            description="Ingest product catalog",
            notebook_task=jobs.NotebookTask(
                notebook_path="/Ingestion/products"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=1
            )
        ),
        
        # === FAN-IN (Merge) ===
        
        # Task 4: Join all data (depends on 1, 2, 3)
        jobs.Task(
            task_key="join_all_data",
            description="Join customers, orders, products",
            depends_on=[
                jobs.TaskDependency(task_key="ingest_customers"),  # ← Wait for 1
                jobs.TaskDependency(task_key="ingest_orders"),     # ← Wait for 2
                jobs.TaskDependency(task_key="ingest_products")    # ← Wait for 3
            ],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Transform/join_data"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.2xlarge",  # Larger for joins
                num_workers=6
            )
        ),
        
        # Task 5: Create analytics (depends on join)
        jobs.Task(
            task_key="create_analytics",
            description="Create analytics tables",
            depends_on=[jobs.TaskDependency(task_key="join_all_data")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Analytics/create_tables"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=3
            )
        )
    ]
)

print(f"Fan-In job created: {job.job_id}")
""")

print("\n" + "="*80)
print("EXECUTION FLOW")
print("="*80)

print("""
When job runs:

1:00:00 - Job starts

        ┌──── PARALLEL (Fan-Out) ────┐
1:00:00 - [Customers] starts
1:00:00 - [Orders] starts
1:00:00 - [Products] starts
        └──────────────────────────┘

1:05:00 - [Products] completes (5 min)
1:08:00 - [Customers] completes (8 min)
1:10:00 - [Orders] completes (10 min) ← Last one!

        ┌──── WAIT FOR ALL ────┐
1:10:00 - [Join All Data] starts (all deps met)
        └───────────────────────┘

1:25:00 - [Join All Data] completes (15 min)
1:25:00 - [Create Analytics] starts
1:35:00 - [Create Analytics] completes (10 min)
1:35:00 - Job completes ✓

Total time: 35 minutes
""")

print("\n" + "="*80)
print("KEY OBSERVATIONS")
print("="*80)

print("""
1. PARALLEL START
   • Tasks 1, 2, 3 start simultaneously (no dependencies)
   • Run in parallel for 10 minutes

2. SYNCHRONIZATION POINT
   • Task 4 (Join) waits for ALL upstream tasks
   • Doesn't start until longest task completes
   • This is the "Fan-In" or "Merge"

3. MULTIPLE DEPENDENCIES
   • Task 4 has 3 dependencies: [1, 2, 3]
   • ALL must succeed for Task 4 to start
   • If any fails, Task 4 never runs

4. SEQUENTIAL AFTER MERGE
   • After merge, execution is sequential
   • Task 5 waits for Task 4
""")

print("\n" + "="*80)
print("PROS AND CONS")
print("="*80)

print("""
✓ ADVANTAGES:
  • Combines speed of parallel + data consolidation
  • Efficient for multi-source ingestion
  • Natural pattern for data integration
  • Ensures all data available before joining

✗ DISADVANTAGES:
  • Slower than pure parallel (has wait point)
  • Bottlenecked by slowest upstream task
  • Merge task may need large cluster
  • If one upstream fails, merge never runs
""")

print("\n" + "="*80)
print("WHEN TO USE FAN-IN")
print("="*80)

print("""
✓ Use Fan-In when:
  • Need to combine data from multiple sources
  • Sources can be ingested in parallel
  • Downstream task needs ALL upstream data
  • Building data warehouse / lake house
  • Creating unified customer view

✗ Don't use when:
  • Only one data source
  • Sources depend on each other (use sequential)
  • Don't need to combine data
""")

print("\n" + "="*80)
print("ERROR HANDLING IN FAN-IN")
print("="*80)

print("""
What if one upstream task fails?

Scenario: Orders ingestion fails

1:00:00 - All 3 tasks start
1:05:00 - [Products] completes ✓
1:08:00 - [Customers] completes ✓
1:10:00 - [Orders] fails ✗ (after retries)
1:10:00 - Job FAILS

Result:
  • [Join All Data] NEVER runs (missing dependency)
  • [Create Analytics] NEVER runs
  • Job fails even though 2/3 tasks succeeded

Best Practice:
  • Implement robust retry logic
  • Consider partial processing (advanced)
  • Monitor upstream tasks closely
  • Have fallback/default data if non-critical
""")

print("\n" + "="*80)
print("PARAMETER PASSING IN FAN-IN")
print("="*80)

print("""
# Upstream tasks output metadata

# Task 1: Customers
dbutils.jobs.taskValues.set("customer_count", df.count())

# Task 2: Orders  
dbutils.jobs.taskValues.set("order_count", df.count())

# Task 3: Products
dbutils.jobs.taskValues.set("product_count", df.count())


# Downstream merge task reads all

# Task 4: Join All Data
customer_count = dbutils.jobs.taskValues.get("ingest_customers", "customer_count")
order_count = dbutils.jobs.taskValues.get("ingest_orders", "order_count")
product_count = dbutils.jobs.taskValues.get("ingest_products", "product_count")

print(f"Joining {customer_count} customers, {order_count} orders, {product_count} products")
""")

print("\n" + "="*80)
print("REAL-WORLD EXAMPLE: DATA WAREHOUSE LOAD")
print("="*80)

print("""
Scenario: Nightly data warehouse refresh

Parallel Ingestion (20 min):
  • Ingest sales transactions from POS system
  • Ingest customer updates from CRM
  • Ingest product catalog from ERP
  • Ingest web clickstream from S3

Merge (30 min):
  • Join all sources on common keys
  • Apply business logic
  • Create fact and dimension tables

Final (10 min):
  • Update aggregated summary tables
  • Refresh BI dashboard cache

Total: 60 minutes (vs 120 min sequential)
""")

print("\n" + "="*80)
print("END OF FAN-IN ORCHESTRATION")
print("="*80)

In [0]:
# ============================================
# STEP 4: CONDITIONAL ORCHESTRATION
# ============================================
# Pattern: Task A → Decision → Task B (if condition)
#                            └→ Task C (else)
# Different paths based on results

print("="*80)
print("CONDITIONAL ORCHESTRATION - IF-THEN-ELSE PATTERN")
print("="*80)

print("""
Use Case: Data Quality Gate
  Task 1: Load new data
  Task 2: Run quality checks
  Task 3: If quality passed → Promote to production
  Task 4: If quality failed → Quarantine and alert

Execution Flow:
  [Load Data] → [Quality Check] ─┬→ [Promote] (if passed)
                                  └→ [Quarantine] (if failed)
""")

print("\n" + "="*80)
print("CODE: Conditional Job with If-Else")
print("="*80)

print("""
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient()

job = w.jobs.create(
    name="Conditional Data Quality Pipeline",
    tasks=[
        # Task 1: Load staging data
        jobs.Task(
            task_key="load_staging",
            description="Load data to staging table",
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipeline/01_load_staging"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=2
            )
        ),
        
        # Task 2: Run quality checks (returns pass/fail)
        jobs.Task(
            task_key="quality_check",
            description="Validate data quality",
            depends_on=[jobs.TaskDependency(task_key="load_staging")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipeline/02_quality_check"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=1
            )
        ),
        
        # Task 3: Promote to production (runs if quality passed)
        jobs.Task(
            task_key="promote_to_prod",
            description="Promote validated data to production",
            depends_on=[
                jobs.TaskDependency(
                    task_key="quality_check",
                    outcome="true"  # ← CONDITIONAL: Only if succeeded
                )
            ],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipeline/03_promote_production"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=2
            )
        ),
        
        # Task 4: Quarantine bad data (runs if quality failed)
        jobs.Task(
            task_key="quarantine_data",
            description="Move failed data to quarantine",
            depends_on=[
                jobs.TaskDependency(
                    task_key="quality_check",
                    outcome="false"  # ← CONDITIONAL: Only if failed
                )
            ],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipeline/03_quarantine"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=1
            )
        ),
        
        # Task 5: Send alert (runs if quality failed)
        jobs.Task(
            task_key="send_alert",
            description="Alert team about quality issues",
            depends_on=[
                jobs.TaskDependency(task_key="quarantine_data")
            ],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipeline/04_send_alert"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=1
            )
        )
    ]
)

print(f"Conditional job created: {job.job_id}")
""")

print("\n" + "="*80)
print("QUALITY CHECK NOTEBOOK LOGIC")
print("="*80)

print("""
# Notebook: 02_quality_check

from pyspark.sql.functions import *

# Load staging data
df = spark.table("staging.customer_data")

# Define quality checks
null_count = df.filter(col("customer_id").isNull()).count()
duplicate_count = df.groupBy("customer_id").count().filter("count > 1").count()
invalid_email = df.filter(~col("email").rlike("^[^@]+@[^@]+$")).count()

# Calculate quality score
total_rows = df.count()
quality_score = 1 - ((null_count + duplicate_count + invalid_email) / (total_rows * 3))

# Log results
dbutils.jobs.taskValues.set("quality_score", quality_score)
dbutils.jobs.taskValues.set("null_count", null_count)
dbutils.jobs.taskValues.set("duplicate_count", duplicate_count)
dbutils.jobs.taskValues.set("invalid_email", invalid_email)

print(f"Quality Score: {quality_score:.2%}")
print(f"Null IDs: {null_count}")
print(f"Duplicates: {duplicate_count}")
print(f"Invalid Emails: {invalid_email}")

# Decision: Pass or Fail
QUALITY_THRESHOLD = 0.95

if quality_score >= QUALITY_THRESHOLD:
    print(f"✓ PASSED: Quality score {quality_score:.2%} >= {QUALITY_THRESHOLD:.2%}")
    dbutils.notebook.exit("PASSED")  # Exit with success
else:
    print(f"✗ FAILED: Quality score {quality_score:.2%} < {QUALITY_THRESHOLD:.2%}")
    # Don't exit with error - let conditional routing handle it
    dbutils.notebook.exit("FAILED")  # Exit with success but FAILED status
""")

print("\n" + "="*80)
print("EXECUTION FLOW - SUCCESS SCENARIO")
print("="*80)

print("""
Scenario: Data passes quality checks

2:00 AM - [Load Staging] starts
2:05 AM - [Load Staging] completes ✓
2:05 AM - [Quality Check] starts
2:10 AM - [Quality Check] completes ✓ (96% quality score)

        ┌── DECISION: PASSED ──┐
2:10 AM - [Promote to Prod] starts  │
2:10 AM - [Quarantine] SKIPPED      │
2:10 AM - [Send Alert] SKIPPED      │
        └─────────────────────┘

2:20 AM - [Promote to Prod] completes ✓
2:20 AM - Job completes ✓

Result: Data in production!
""")

print("\n" + "="*80)
print("EXECUTION FLOW - FAILURE SCENARIO")
print("="*80)

print("""
Scenario: Data fails quality checks

2:00 AM - [Load Staging] starts
2:05 AM - [Load Staging] completes ✓
2:05 AM - [Quality Check] starts
2:10 AM - [Quality Check] completes (85% quality score - FAILED)

        ┌── DECISION: FAILED ──┐
2:10 AM - [Promote to Prod] SKIPPED   │
2:10 AM - [Quarantine] starts          │
        └───────────────────────┘

2:15 AM - [Quarantine] completes ✓
2:15 AM - [Send Alert] starts
2:16 AM - [Send Alert] completes ✓
2:16 AM - Job completes ✓ (but data NOT promoted)

Result: Data quarantined, team alerted!
""")

print("\n" + "="*80)
print("KEY OBSERVATIONS")
print("="*80)

print("""
1. BRANCHING LOGIC
   • After quality check, execution branches
   • One path for success, another for failure
   • Only ONE path executes

2. TASK SKIPPING
   • Skipped tasks marked as SKIPPED (not FAILED)
   • Job can still succeed with skipped tasks
   • This is different from task failure

3. CONDITIONAL DEPENDENCIES
   • outcome="true" → run if parent succeeded
   • outcome="false" → run if parent failed
   • No outcome → run regardless (default)

4. COMPLEX LOGIC
   • Can have multiple conditional branches
   • Can combine with parallel/fan-in
   • Enables sophisticated workflows
""")

print("\n" + "="*80)
print("PROS AND CONS")
print("="*80)

print("""
✓ ADVANTAGES:
  • Implement business rules in workflows
  • Different handling for success/failure
  • Data quality gates
  • Error handling workflows
  • Flexible routing

✗ DISADVANTAGES:
  • More complex to design and debug
  • Harder to predict execution path
  • Requires careful testing
  • Can become spaghetti if overused
""")

print("\n" + "="*80)
print("WHEN TO USE CONDITIONAL")
print("="*80)

print("""
✓ Use Conditional when:
  • Need data quality gates
  • Different processing for different scenarios
  • Error handling with recovery paths
  • Business rules require branching
  • A/B testing or feature flags

✗ Don't use when:
  • Simple linear pipeline
  • All tasks always run
  • Logic can be in notebook (simpler)
  • Makes workflow too complex
""")

print("\n" + "="*80)
print("ADVANCED: Multi-Way Branching")
print("="*80)

print("""
You can have more than 2 branches!

Scenario: Process based on data volume

[Load Data] → [Check Size] ─┬→ [Small Pipeline] (< 1GB)
                             ├→ [Medium Pipeline] (1-10GB)
                             └→ [Large Pipeline] (> 10GB)

Implementation:
  1. Check Size task outputs: SMALL, MEDIUM, or LARGE
  2. Use task values to set flags
  3. Each pipeline has conditional dependency
  4. Only matching pipeline runs
""")

print("\n" + "="*80)
print("REAL-WORLD EXAMPLE: ML MODEL DEPLOYMENT")
print("="*80)

print("""
Scenario: Train and conditionally deploy model

[Train Model] → [Evaluate] ─┬→ [Deploy] (if better than current)
                            └→ [Archive] (if not better)

Evaluation Logic:
  • Compare new model accuracy to production model
  • If new model > production + 2%: DEPLOY
  • Else: ARCHIVE for future analysis

Result:
  • Only better models reach production
  • Automatic quality gate
  • Safe automated deployment
""")

print("\n" + "="*80)
print("END OF CONDITIONAL ORCHESTRATION")
print("="*80)

In [0]:
# ============================================
# STEP 1: SEQUENTIAL ORCHESTRATION
# ============================================
# Pattern: Task A → Task B → Task C → Task D
# Each task waits for the previous to complete

print("="*80)
print("SEQUENTIAL ORCHESTRATION - LINEAR PIPELINE")
print("="*80)

print("""
Use Case: ETL Pipeline
  Step 1: Extract data from source
  Step 2: Transform and clean (needs Step 1 output)
  Step 3: Load to data warehouse (needs Step 2 output)
  Step 4: Update dashboard (needs Step 3 output)

Execution Flow:
  [Extract] → [Transform] → [Load] → [Dashboard]
  
Time: Sequential (4 tasks × 10 min each = 40 minutes total)
""")

print("\n" + "="*80)
print("CODE: Create Sequential Job")
print("="*80)

print("""
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient()

# Create job with sequential tasks
job = w.jobs.create(
    name="Sequential ETL Pipeline",
    tasks=[
        # Task 1: Extract (no dependencies)
        jobs.Task(
            task_key="extract",
            description="Extract data from source",
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipelines/01_extract",
                base_parameters={"source": "s3://bucket/raw/"}
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=2
            ),
            timeout_seconds=3600,  # 1 hour
            max_retries=2
        ),
        
        # Task 2: Transform (depends on extract)
        jobs.Task(
            task_key="transform",
            description="Transform and clean data",
            depends_on=[jobs.TaskDependency(task_key="extract")],  # ← DEPENDENCY
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipelines/02_transform"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=4  # More workers for transformation
            ),
            timeout_seconds=3600,
            max_retries=2
        ),
        
        # Task 3: Load (depends on transform)
        jobs.Task(
            task_key="load",
            description="Load to warehouse",
            depends_on=[jobs.TaskDependency(task_key="transform")],  # ← DEPENDENCY
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipelines/03_load",
                base_parameters={"target": "main.warehouse.sales"}
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=2
            ),
            timeout_seconds=1800,
            max_retries=3  # More retries for critical load step
        ),
        
        # Task 4: Update Dashboard (depends on load)
        jobs.Task(
            task_key="update_dashboard",
            description="Refresh dashboard data",
            depends_on=[jobs.TaskDependency(task_key="load")],  # ← DEPENDENCY
            sql_task=jobs.SqlTask(
                warehouse_id="<warehouse-id>",
                query=jobs.SqlTaskQuery(
                    query_id="<query-id>"  # Saved query
                )
            ),
            timeout_seconds=600
        )
    ],
    
    # Schedule: Daily at 2 AM
    schedule=jobs.CronSchedule(
        quartz_cron_expression="0 0 2 * * ? *",
        timezone_id="UTC"
    ),
    
    # Notifications
    email_notifications=jobs.JobEmailNotifications(
        on_failure=["data-team@company.com"],
        on_success=["data-team@company.com"]
    ),
    
    # Job-level settings
    max_concurrent_runs=1,  # Only one run at a time
    timeout_seconds=14400   # 4 hour max for entire job
)

print(f"Sequential job created: {job.job_id}")
""")

print("\n" + "="*80)
print("EXECUTION FLOW")
print("="*80)

print("""
When job runs:

2:00 AM - Job starts
2:00 AM - [Extract] starts
2:10 AM - [Extract] completes successfully
2:10 AM - [Transform] starts (waited for Extract)
2:20 AM - [Transform] completes successfully
2:20 AM - [Load] starts (waited for Transform)
2:30 AM - [Load] completes successfully
2:30 AM - [Update Dashboard] starts (waited for Load)
2:32 AM - [Update Dashboard] completes
2:32 AM - Job completes ✓

Total time: 32 minutes
""")

print("\n" + "="*80)
print("PROS AND CONS")
print("="*80)

print("""
✓ ADVANTAGES:
  • Simple to understand and debug
  • Clear execution order
  • Easy to track data lineage
  • Predictable resource usage

✗ DISADVANTAGES:
  • Slow - no parallelization
  • Idle time if one task is fast
  • Total time = sum of all task times
  • Not efficient for independent tasks
""")

print("\n" + "="*80)
print("WHEN TO USE SEQUENTIAL")
print("="*80)

print("""
✓ Use Sequential when:
  • Each task needs output from previous task
  • Tasks must run in specific order
  • Simple, linear data flow
  • Limited compute resources
  • Debugging simplicity is priority

✗ Don't use when:
  • Tasks are independent (use parallel)
  • Speed is critical (use parallel)
  • Have multiple data sources (use fan-in)
""")

print("\n" + "="*80)
print("ERROR HANDLING IN SEQUENTIAL")
print("="*80)

print("""
What happens if a task fails?

Scenario: Transform task fails

2:00 AM - [Extract] completes ✓
2:10 AM - [Transform] starts
2:12 AM - [Transform] fails ✗
2:12 AM - [Transform] retry 1 starts
2:14 AM - [Transform] retry 1 fails ✗
2:14 AM - [Transform] retry 2 starts
2:16 AM - [Transform] retry 2 fails ✗
2:16 AM - Job FAILS - downstream tasks NOT executed
2:16 AM - Email alert sent to data-team@company.com

Result:
  • [Load] never runs
  • [Update Dashboard] never runs
  • Entire job marked as FAILED
  • Manual intervention required
""")

print("\n" + "="*80)
print("PARAMETER PASSING BETWEEN TASKS")
print("="*80)

print("""
# Task 1: Extract - output parameters
notebook code:
  record_count = df.count()
  dbutils.jobs.taskValues.set(key="record_count", value=record_count)
  dbutils.notebook.exit(record_count)

# Task 2: Transform - read parameters from Task 1
notebook code:
  from databricks.sdk.runtime import *
  record_count = dbutils.jobs.taskValues.get(
      taskKey="extract",
      key="record_count",
      debugValue=0
  )
  print(f"Processing {record_count} records from extract task")
""")

print("\n" + "="*80)
print("END OF SEQUENTIAL ORCHESTRATION")
print("="*80)

In [0]:
# ============================================
# STEP 2: PARALLEL ORCHESTRATION (FAN-OUT)
# ============================================
# Pattern: Task A ─┬→ Task B
#                 ├→ Task C
#                 └→ Task D
# Multiple tasks run simultaneously

print("="*80)
print("PARALLEL ORCHESTRATION - FAN-OUT PATTERN")
print("="*80)

print("""
Use Case: Multi-Source Data Ingestion
  Trigger: Start ingestion
  Task 1: Ingest from S3 (parallel)
  Task 2: Ingest from Snowflake (parallel)
  Task 3: Ingest from PostgreSQL (parallel)
  Task 4: Ingest from API (parallel)

Execution Flow:
                 ─┬→ [S3]
                  ├→ [Snowflake]
  [Trigger] ─────├→ [PostgreSQL]
                  └→ [API]
  
All ingest tasks run AT THE SAME TIME!
Time: Parallel (longest task = 15 min, not 4 × 15 = 60 min)
""")

print("\n" + "="*80)
print("CODE: Create Parallel Job")
print("="*80)

print("""
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient()

job = w.jobs.create(
    name="Parallel Multi-Source Ingestion",
    tasks=[
        # Trigger task (runs first)
        jobs.Task(
            task_key="trigger",
            description="Initialize ingestion process",
            notebook_task=jobs.NotebookTask(
                notebook_path="/Ingestion/00_trigger"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=1
            )
        ),
        
        # Task 1: S3 ingestion (depends only on trigger)
        jobs.Task(
            task_key="ingest_s3",
            description="Ingest data from S3",
            depends_on=[jobs.TaskDependency(task_key="trigger")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Ingestion/ingest_s3",
                base_parameters={"bucket": "s3://data-lake/raw/"}
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=4
            )
        ),
        
        # Task 2: Snowflake ingestion (depends only on trigger)
        jobs.Task(
            task_key="ingest_snowflake",
            description="Ingest from Snowflake",
            depends_on=[jobs.TaskDependency(task_key="trigger")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Ingestion/ingest_snowflake"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=3
            )
        ),
        
        # Task 3: PostgreSQL ingestion (depends only on trigger)
        jobs.Task(
            task_key="ingest_postgres",
            description="Ingest from PostgreSQL",
            depends_on=[jobs.TaskDependency(task_key="trigger")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Ingestion/ingest_postgres"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=2
            )
        ),
        
        # Task 4: API ingestion (depends only on trigger)
        jobs.Task(
            task_key="ingest_api",
            description="Ingest from REST API",
            depends_on=[jobs.TaskDependency(task_key="trigger")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Ingestion/ingest_api"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=2
            )
        )
    ],
    
    max_concurrent_runs=1
)

print(f"Parallel job created: {job.job_id}")
""")

print("\n" + "="*80)
print("EXECUTION FLOW")
print("="*80)

print("""
When job runs:

10:00:00 - Job starts
10:00:00 - [Trigger] starts
10:00:30 - [Trigger] completes

           ┌─ ALL START AT SAME TIME (parallel!) ──┐
           │                                        │
10:00:30 - [S3 Ingest] starts                    │
10:00:30 - [Snowflake Ingest] starts             │  PARALLEL
10:00:30 - [Postgres Ingest] starts              │  EXECUTION
10:00:30 - [API Ingest] starts                   │
           │                                        │
           └────────────────────────────────────────┘

10:05:00 - [API Ingest] completes (5 min)
10:10:00 - [Postgres Ingest] completes (10 min)
10:12:00 - [Snowflake Ingest] completes (12 min)
10:15:00 - [S3 Ingest] completes (15 min) ← Longest
10:15:00 - Job completes ✓

Total time: 15 minutes (not 30 + 5 + 10 + 12 + 15 = 72 min!)
Speedup: 72 / 15 = 4.8x faster!
""")

print("\n" + "="*80)
print("KEY OBSERVATIONS")
print("="*80)

print("""
1. PARALLEL EXECUTION
   • All 4 ingest tasks start simultaneously
   • Each runs on its own cluster
   • No waiting for others to complete

2. TOTAL TIME = LONGEST TASK
   • Job completes when slowest task finishes
   • Not sum of all task times
   • Huge time savings!

3. RESOURCE USAGE
   • 4 clusters running at once
   • Higher compute cost during execution
   • But faster completion = less total cost

4. DEPENDENCIES
   • All tasks depend only on 'trigger'
   • No dependencies between parallel tasks
   • That's what allows parallel execution
""")

print("\n" + "="*80)
print("PROS AND CONS")
print("="*80)

print("""
✓ ADVANTAGES:
  • FAST - major time savings
  • Efficient for independent tasks
  • Maximize cluster utilization
  • Reduce end-to-end latency

✗ DISADVANTAGES:
  • Higher compute cost (multiple clusters)
  • May hit cluster quotas/limits
  • Harder to debug (multiple logs)
  • Requires tasks to be independent
""")

print("\n" + "="*80)
print("WHEN TO USE PARALLEL")
print("="*80)

print("""
✓ Use Parallel when:
  • Tasks are INDEPENDENT (no data dependencies)
  • Speed is critical
  • Multiple data sources to process
  • Tasks can run simultaneously
  • Have sufficient compute quota

✗ Don't use when:
  • Tasks depend on each other's output
  • Limited compute resources
  • Tasks share same data source (may overload)
  • Cost is more important than speed
""")

print("\n" + "="*80)
print("ERROR HANDLING IN PARALLEL")
print("="*80)

print("""
What if one parallel task fails?

Scenario: Snowflake ingestion fails

10:00:30 - All 4 tasks start
10:05:00 - [API] completes ✓
10:10:00 - [Postgres] completes ✓
10:12:00 - [Snowflake] fails ✗ (after 2 retries)
10:15:00 - [S3] completes ✓
10:15:00 - Job FAILS (one task failed)

Result:
  • 3 tasks succeeded
  • 1 task failed
  • Entire job marked as FAILED
  • May need to re-run only failed task

Best Practice:
  • Implement idempotent tasks (safe to re-run)
  • Use task-level retry policies
  • Consider using 'continue on failure' for non-critical tasks
""")

print("\n" + "="*80)
print("REAL-WORLD EXAMPLE: ML FEATURE ENGINEERING")
print("="*80)

print("""
Use Case: Compute features from multiple sources in parallel

Tasks:
  1. Compute user demographics (15 min)
  2. Compute purchase history (20 min)
  3. Compute behavioral features (25 min)
  4. Compute social features (10 min)

Sequential time: 15 + 20 + 25 + 10 = 70 minutes
Parallel time:   max(15, 20, 25, 10) = 25 minutes

Savings: 45 minutes (64% faster!)
""")

print("\n" + "="*80)
print("END OF PARALLEL ORCHESTRATION")
print("="*80)

In [0]:
# ============================================
# TASK ORCHESTRATION - Complete Guide
# ============================================
# Orchestration = Coordinating multiple tasks in a workflow

print("="*80)
print("WHAT IS TASK ORCHESTRATION?")
print("="*80)

print("""
Task Orchestration is coordinating multiple steps in a data workflow.

Simple workflow:
  Task 1: Extract data → Task 2: Transform data → Task 3: Load to warehouse

Complex workflow:
  Task 1: Ingest from 3 sources (parallel)
  Task 2: Wait for all to complete
  Task 3: Join and transform
  Task 4: Run data quality checks
  Task 5: If passed → Load to production
  Task 6: If failed → Send alert and quarantine
  Task 7: Generate report and email
""")

print("\n" + "="*80)
print("WHY DO WE NEED ORCHESTRATION?")
print("="*80)

print("""
Real-world data pipelines have:
  ✓ Multiple data sources to process
  ✓ Dependencies between tasks (Task B needs Task A output)
  ✓ Parallel processing opportunities (speed up workflows)
  ✓ Conditional logic (if-then-else)
  ✓ Error handling and retries
  ✓ Scheduling requirements (daily, hourly, on-demand)

Without orchestration:
  ✗ Run tasks manually in order
  ✗ Hard to track failures
  ✗ Difficult to parallelize
  ✗ No automatic retries
  ✗ Complex scheduling

With orchestration:
  ✓ Automatic execution in correct order
  ✓ Dependency management
  ✓ Parallel execution where possible
  ✓ Error handling and retries
  ✓ Monitoring and alerts
""")

print("\n" + "="*80)
print("ORCHESTRATION PATTERNS")
print("="*80)

print("""
┌────────────────────────────────────────────────────────────────┐
│ Pattern 1: SEQUENTIAL (Linear Pipeline)                       │
│   Task A → Task B → Task C → Task D                           │
│   Use: When each task depends on the previous one             │
└────────────────────────────────────────────────────────────────┘

Example: ETL Pipeline
  Task 1: Extract from source
  Task 2: Transform (depends on Task 1)
  Task 3: Load to warehouse (depends on Task 2)
  Task 4: Update dashboard (depends on Task 3)


┌────────────────────────────────────────────────────────────────┐
│ Pattern 2: PARALLEL (Fan-Out)                                 │
│   Task A ─┬→ Task B                                           │
│           ├→ Task C                                            │
│           └→ Task D                                            │
│   Use: When tasks are independent, run simultaneously         │
└────────────────────────────────────────────────────────────────┘

Example: Multi-Source Ingestion
  Task 1: Trigger ingestion
  Task 2: Ingest from S3 (parallel)
  Task 3: Ingest from API (parallel)
  Task 4: Ingest from database (parallel)
  All run at the same time!


┌────────────────────────────────────────────────────────────────┐
│ Pattern 3: FAN-IN (Merge)                                     │
│   Task A ─┐                                                   │
│   Task B ─┤→ Task D                                           │
│   Task C ─┘                                                   │
│   Use: Wait for multiple tasks before proceeding              │
└────────────────────────────────────────────────────────────────┘

Example: Data Consolidation
  Task 1: Ingest customers (parallel)
  Task 2: Ingest orders (parallel)
  Task 3: Ingest products (parallel)
  Task 4: Join all data (waits for 1, 2, 3)


┌────────────────────────────────────────────────────────────────┐
│ Pattern 4: CONDITIONAL (If-Then-Else)                         │
│   Task A → Decision → Task B (if condition)                   │
│                    └→ Task C (else)                           │
│   Use: Different paths based on results                       │
└────────────────────────────────────────────────────────────────┘

Example: Data Quality Check
  Task 1: Load data
  Task 2: Run quality checks
  Task 3: If passed → Promote to production
  Task 4: If failed → Quarantine and alert


┌────────────────────────────────────────────────────────────────┐
│ Pattern 5: DIAMOND (Parallel + Merge)                         │
│   Task A ─┬→ Task B ─┐                                        │
│           └→ Task C ─┤→ Task D                                │
│   Use: Parallel processing then combine                       │
└────────────────────────────────────────────────────────────────┘

Example: ML Feature Engineering
  Task 1: Load raw data
  Task 2: Compute numerical features (parallel)
  Task 3: Compute categorical features (parallel)
  Task 4: Combine features and train model
""")

print("\n" + "="*80)
print("DATABRICKS ORCHESTRATION TOOLS")
print("="*80)

print("""
1. DATABRICKS JOBS (Workflows)
   ✓ Native Databricks orchestration
   ✓ Task dependencies with drag-and-drop UI
   ✓ Multiple task types (notebooks, Python, SQL, DLT)
   ✓ Built-in monitoring and alerts
   ✓ Retry policies and error handling
   ✓ Best for: Most Databricks workloads

2. DELTA LIVE TABLES (DLT)
   ✓ Auto-orchestration for data pipelines
   ✓ Dependencies inferred from code
   ✓ Built-in data quality
   ✓ Best for: Bronze-Silver-Gold pipelines

3. EXTERNAL TOOLS (Airflow, Prefect, etc.)
   ✓ Organization-wide orchestration
   ✓ Multi-platform workflows
   ✓ Advanced DAG features
   ✓ Best for: Cross-platform orchestration
""")

print("\n" + "="*80)
print("KEY ORCHESTRATION CONCEPTS")
print("="*80)

print("""
1. TASK
   • Single unit of work (run a notebook, execute SQL, etc.)
   • Has inputs, outputs, and success/failure state

2. DEPENDENCY
   • Task B depends on Task A = Task B waits for Task A to complete
   • Can depend on multiple tasks (fan-in)

3. DAG (Directed Acyclic Graph)
   • Visual representation of workflow
   • Nodes = Tasks, Edges = Dependencies
   • Acyclic = No circular dependencies (Task A → B → C → A ✗)

4. PARALLEL EXECUTION
   • Tasks with no dependencies run simultaneously
   • Speeds up workflows

5. RETRY POLICY
   • Automatically retry failed tasks
   • Max retries, retry interval

6. TIMEOUT
   • Maximum execution time for a task
   • Prevents hanging jobs

7. PARAMETERS
   • Pass data between tasks
   • Example: Task A outputs date, Task B uses that date
""")

print("\n" + "="*80)
print("ORCHESTRATION LIFECYCLE")
print("="*80)

print("""
Step 1: DEFINE WORKFLOW
  • Create tasks
  • Set dependencies
  • Configure retries and timeouts

Step 2: SCHEDULE
  • Cron expression (daily, hourly, etc.)
  • Event trigger (file arrival)
  • Manual (on-demand)

Step 3: EXECUTE
  • Workflow engine starts
  • Tasks run in dependency order
  • Parallel tasks run simultaneously

Step 4: MONITOR
  • View task status (pending, running, success, failed)
  • Check logs and metrics
  • Receive alerts on failures

Step 5: HANDLE ERRORS
  • Automatic retries
  • Alerts to on-call team
  • Manual intervention if needed
""")

print("\n" + "="*80)
print("WHEN TO USE EACH PATTERN")
print("="*80)

print("""
SEQUENTIAL:
  ✓ Simple ETL pipelines
  ✓ Each step needs previous output
  ✗ Can be slow (no parallelization)

PARALLEL:
  ✓ Independent data sources
  ✓ Maximize throughput
  ✓ Reduce total execution time
  ✗ May need more compute resources

FAN-IN:
  ✓ Consolidate data from multiple sources
  ✓ Ensure all upstream tasks complete
  ✓ Common in data warehousing

CONDITIONAL:
  ✓ Data quality gates
  ✓ Different processing for different data
  ✓ Error handling paths
  ✗ More complex to debug

DIAMOND:
  ✓ Balance parallelism and synchronization
  ✓ ML feature engineering
  ✓ Multi-stage transformations
""")

print("\n" + "="*80)
print("NEXT: See step-by-step examples with code")
print("="*80)

In [0]:
# ============================================
# DLT vs JOBS - When to Use What?
# ============================================

print("="*80)
print("DELTA LIVE TABLES vs DATABRICKS JOBS - COMPLETE COMPARISON")
print("="*80)

print("\n" + "="*80)
print("FEATURE COMPARISON")
print("="*80)

print("\n{:<30} {:<25} {:<25}".format("Feature", "Delta Live Tables", "Databricks Jobs"))
print("-" * 80)

comparisons = [
    ("Purpose", "Data pipelines", "General workflows"),
    ("Code Style", "Declarative", "Imperative"),
    ("Dependencies", "Auto-inferred", "Manual definition"),
    ("Data Quality", "Built-in expectations", "Manual implementation"),
    ("Lineage", "Automatic", "Manual tracking"),
    ("Monitoring", "Built-in UI", "Basic monitoring"),
    ("Language", "Python, SQL", "Python, SQL, Scala, R"),
    ("Task Types", "Table definitions", "Notebooks, scripts, JARs"),
    ("Scheduling", "Triggered/Continuous", "Cron, triggers, manual"),
    ("Complexity", "Simplified", "More flexible"),
    ("Cost", "Higher (managed)", "Lower (DIY)"),
    ("Best For", "ETL pipelines", "Any workflow")
]

for feature, dlt, jobs in comparisons:
    print("{:<30} {:<25} {:<25}".format(feature, dlt, jobs))

print("\n" + "="*80)
print("DECISION TREE: WHICH SHOULD I USE?")
print("="*80)

print("""
┌─────────────────────────────────────┐
│   Are you building a data pipeline?   │
└─────────────────────────────────────┘
           │
           ├─── YES → Bronze/Silver/Gold architecture?
           │            │
           │            ├─── YES → Need data quality checks?
           │            │            │
           │            │            ├─── YES → 🟢 USE DLT
           │            │            │
           │            │            └─── NO  → 🟡 EITHER (DLT preferred)
           │            │
           │            └─── NO  → Simple ETL?
           │                         │
           │                         ├─── YES → 🟡 EITHER
           │                         │
           │                         └─── NO  → 🔵 USE JOBS
           │
           └─── NO  → ML training? Report generation?
                        │
                        └─── YES → 🔵 USE JOBS
""")

print("\n" + "="*80)
print("USE CASES: DLT")
print("="*80)

print("""
✓ PERFECT FOR:
  • Bronze-Silver-Gold data pipelines
  • Streaming ETL with Auto Loader
  • Data quality enforcement (expectations)
  • Complex transformation chains
  • CDC (Change Data Capture) pipelines
  • Real-time analytics preparation
  • Regulatory compliance (lineage tracking)

Example: Customer 360 pipeline
  Bronze: Ingest from Kafka, S3, databases
  Silver: Clean, deduplicate, join
  Gold:   Customer profiles, metrics
""")

print("\n" + "="*80)
print("USE CASES: JOBS")
print("="*80)

print("""
✓ PERFECT FOR:
  • ML model training workflows
  • Report generation and distribution
  • Data export to external systems
  • Complex orchestration (if-then-else logic)
  • Scheduled maintenance tasks
  • Multi-step processes with custom logic
  • Running DLT pipelines on a schedule

Example: Daily ML pipeline
  Task 1: Feature engineering (notebook)
  Task 2: Train model (Python script)
  Task 3: Evaluate model (notebook)
  Task 4: Deploy if better (conditional)
  Task 5: Send report (notebook)
""")

print("\n" + "="*80)
print("CAN YOU USE BOTH TOGETHER?")
print("="*80)

print("""
YES! Common pattern:

  1. DLT Pipeline: Bronze → Silver → Gold
     (Handles data transformation)

  2. Databricks Job: Orchestrates everything
     Task 1: Run DLT pipeline
     Task 2: Wait for completion
     Task 3: Run ML model on Gold tables
     Task 4: Generate reports
     Task 5: Send notifications

Best of both worlds:
  • DLT for data pipelines (quality + lineage)
  • Jobs for orchestration and ML
""")

print("\n" + "="*80)
print("MIGRATION PATH")
print("="*80)

print("""
Starting with Jobs?
  → If building Bronze-Silver-Gold, migrate to DLT
  → Keep Jobs for orchestration and ML

Starting fresh?
  → Use DLT for data pipelines
  → Use Jobs for everything else
  → Combine them for complex workflows
""")

print("\n" + "="*80)
print("REAL-WORLD EXAMPLE: E-commerce Platform")
print("="*80)

print("""
Scenario: Daily analytics for online store

SOLUTION ARCHITECTURE:

1. DLT Pipeline: "customer_data_pipeline"
   Bronze:
     • Raw orders from S3
     • Raw user events from Kafka
     • Raw products from MySQL
   
   Silver:
     • Cleaned orders (dedupe, validate)
     • Sessionized events
     • Product catalog
   
   Gold:
     • Customer 360 view
     • Product performance
     • Daily sales metrics

2. Databricks Job: "daily_analytics_workflow"
   Schedule: Daily at 2 AM
   
   Task 1: Run DLT pipeline (refresh tables)
   Task 2: Train recommendation model (Python)
   Task 3: Score customers (notebook)
   Task 4: Update dashboard tables (SQL)
   Task 5: Send executive report (notebook)
   Task 6: Alert on anomalies (Python)

Result:
  ✓ Data quality ensured by DLT
  ✓ Complex workflow orchestrated by Job
  ✓ ML and reporting seamlessly integrated
""")

print("\n" + "="*80)
print("QUICK DECISION GUIDE")
print("="*80)

print("""
Choose DLT if:
  ✓ Building data pipelines
  ✓ Need built-in quality checks
  ✓ Want automatic lineage
  ✓ Using Bronze-Silver-Gold
  ✓ Processing streaming data

Choose Jobs if:
  ✓ ML model training
  ✓ Complex orchestration
  ✓ Report generation
  ✓ Custom workflows
  ✓ Need if-then-else logic

Use Both if:
  ✓ DLT for data pipeline
  ✓ Job to orchestrate DLT + ML + reporting
""")

print("\n" + "="*80)
print("END OF DLT vs JOBS COMPARISON")
print("="*80)

In [0]:
# ============================================
# CHANGE DATA CAPTURE (CDC) - Complete Guide
# ============================================

print("="*80)
print("WHAT IS CHANGE DATA CAPTURE (CDC)?")
print("="*80)

print("""
CDC = Tracking changes in source data and applying them to target tables.

Without CDC:
  • Load ENTIRE table every time (full refresh)
  • Slow, expensive, resource-intensive
  • Example: 1 billion rows, only 1000 changed → still process 1 billion

With CDC:
  • Track only CHANGES (inserts, updates, deletes)
  • Process only what changed
  • Example: 1 billion rows, 1000 changed → process only 1000
  • Result: Faster, cheaper, more efficient
""")

print("\n" + "="*80)
print("WHY USE CDC?")
print("="*80)

print("""
✓ PERFORMANCE
  • Process only changes, not entire dataset
  • 100x-1000x faster for large tables

✓ COST EFFICIENCY
  • Less compute required
  • Lower data transfer costs

✓ REAL-TIME ANALYTICS
  • Near real-time updates
  • Fresh data without full reloads

✓ COMPLIANCE
  • Track data lineage
  • Audit trail of all changes
  • GDPR/CCPA compliance (track deletions)

✓ RESOURCE OPTIMIZATION
  • Smaller processing windows
  • Less impact on source systems
""")

print("\n" + "="*80)
print("DATA SYNC PATTERNS")
print("="*80)

print("""
┌────────────────────────────────────────────────────────────────┐
│ Pattern 1: FULL LOAD (Snapshot)                               │
│   Load ALL data every time                                    │
└────────────────────────────────────────────────────────────────┘

Pros:
  ✓ Simple to implement
  ✓ Always consistent
  ✓ No change tracking needed

Cons:
  ✗ Slow for large tables
  ✗ Expensive compute costs
  ✗ High network transfer
  ✗ Impacts source systems

Use When:
  • Small tables (< 1 million rows)
  • Data changes frequently (> 50%)
  • Source has no change tracking


┌────────────────────────────────────────────────────────────────┐
│ Pattern 2: INCREMENTAL LOAD (Append-Only)                     │
│   Load only NEW records since last run                        │
└────────────────────────────────────────────────────────────────┘

Pros:
  ✓ Fast (only new data)
  ✓ Simple logic
  ✓ Low cost

Cons:
  ✗ Doesn't handle UPDATES
  ✗ Doesn't handle DELETES
  ✗ Multiple versions of same record

Use When:
  • Append-only data (logs, events, transactions)
  • No updates to existing records
  • Timestamp/ID column available


┌────────────────────────────────────────────────────────────────┐
│ Pattern 3: CDC (Change Data Capture)                          │
│   Track ALL changes: inserts, updates, deletes                │
└────────────────────────────────────────────────────────────────┘

Pros:
  ✓ Captures all changes
  ✓ Fast (only changes)
  ✓ Maintains current state
  ✓ Complete audit trail

Cons:
  ✗ More complex
  ✗ Requires change tracking
  ✗ Merge logic needed

Use When:
  • Large tables with updates/deletes
  • Need current state + history
  • Performance critical
  • Compliance requirements
""")

print("\n" + "="*80)
print("CDC IMPLEMENTATION APPROACHES")
print("="*80)

print("""
1. TIMESTAMP-BASED CDC (Simple)
   • Source table has 'updated_at' column
   • Query: WHERE updated_at > last_load_time
   • Pros: Easy, no special tools
   • Cons: Misses deletes, clock skew issues

2. VERSION-BASED CDC
   • Source table has 'version' column
   • Query: WHERE version > last_version
   • Pros: No clock dependencies
   • Cons: Source must maintain versions

3. LOG-BASED CDC (Production)
   • Read database transaction logs
   • Tools: Debezium, AWS DMS, Fivetran
   • Pros: Captures everything, minimal impact
   • Cons: Requires special setup

4. TRIGGER-BASED CDC
   • Database triggers write changes to audit table
   • Read audit table for changes
   • Pros: Reliable, captures all changes
   • Cons: Performance impact on source

5. DELTA LAKE CDC (Databricks)
   • Use Delta Lake's built-in CDC
   • table_changes() function
   • Pros: Native, efficient, automatic
   • Cons: Only for Delta tables
""")

print("\n" + "="*80)
print("CDC WORKFLOW")
print("="*80)

print("""
Step 1: CAPTURE CHANGES
  • Identify changed records in source
  • Methods: timestamp, version, logs, triggers

Step 2: EXTRACT CHANGES
  • Pull only changed records
  • Include operation type (INSERT, UPDATE, DELETE)

Step 3: TRANSFORM (Optional)
  • Clean, validate, enrich
  • Business logic

Step 4: APPLY CHANGES (MERGE)
  • MERGE statement to apply changes:
    - MATCHED → UPDATE
    - NOT MATCHED → INSERT
    - MATCHED + DELETE flag → DELETE

Step 5: TRACK PROGRESS
  • Save checkpoint (last processed timestamp/version)
  • Next run starts from checkpoint
""")

print("\n" + "="*80)
print("NEXT: See CDC code examples")
print("="*80)

In [0]:
# ============================================
# CDC EXAMPLE 1: Timestamp-Based CDC
# ============================================
# Simplest CDC: Use updated_at timestamp

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, lit
from delta.tables import DeltaTable

print("="*70)
print("SCENARIO: Customer table with updates")
print("="*70)

print("""
Source: customers table in MySQL/Postgres
Columns:
  - customer_id (primary key)
  - name
  - email
  - status
  - updated_at (timestamp of last change)

Goal: Sync only changed customers to Delta Lake
""")

print("\n--- Step 1: Create Sample Source Data ---\n")

# Simulate source table
source_data = [
    (1, "Alice", "alice@email.com", "active", "2024-01-01 10:00:00"),
    (2, "Bob", "bob@email.com", "active", "2024-01-01 10:00:00"),
    (3, "Charlie", "charlie@email.com", "inactive", "2024-01-02 11:00:00"),
    (4, "Diana", "diana@email.com", "active", "2024-01-03 12:00:00"),
]

source_df = spark.createDataFrame(
    source_data,
    ["customer_id", "name", "email", "status", "updated_at"]
)

print("Source customers table:")
display(source_df)

print("\n--- Step 2: Initial Full Load (First Time) ---\n")

# First time: load all data
target_path = "/tmp/cdc_example/customers"
source_df.write.format("delta").mode("overwrite").save(target_path)

print(f"Initial load complete: {source_df.count()} rows")
print("Last sync timestamp: 2024-01-03 12:00:00")

print("\n--- Step 3: Simulate Changes in Source ---\n")

# New data in source (changes after 2024-01-03 12:00:00)
changed_data = [
    # Existing customers - no change
    (1, "Alice", "alice@email.com", "active", "2024-01-01 10:00:00"),
    (2, "Bob", "bob@email.com", "active", "2024-01-01 10:00:00"),
    (3, "Charlie", "charlie@email.com", "inactive", "2024-01-02 11:00:00"),
    (4, "Diana", "diana@email.com", "active", "2024-01-03 12:00:00"),
    # UPDATED: Bob changed email and status
    (2, "Bob Smith", "bob.smith@email.com", "inactive", "2024-01-04 14:00:00"),
    # NEW: Eve joined
    (5, "Eve", "eve@email.com", "active", "2024-01-04 15:00:00"),
]

new_source_df = spark.createDataFrame(
    changed_data,
    ["customer_id", "name", "email", "status", "updated_at"]
)

print("Updated source table:")
display(new_source_df.orderBy("customer_id", "updated_at"))

print("\n--- Step 4: CDC - Extract Only Changes ---\n")

last_sync_time = "2024-01-03 12:00:00"

# Extract only changed records
changes_df = new_source_df.filter(col("updated_at") > lit(last_sync_time))

print(f"Changes detected: {changes_df.count()} records")
print("Changed records:")
display(changes_df)

print("\n--- Step 5: Apply Changes Using MERGE ---\n")

# Load target Delta table
target_table = DeltaTable.forPath(spark, target_path)

# MERGE logic:
# - If customer_id matches → UPDATE
# - If customer_id doesn't match → INSERT
target_table.alias("target").merge(
    changes_df.alias("source"),
    "target.customer_id = source.customer_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print("Merge complete!")
print("\nFinal target table:")
final_df = spark.read.format("delta").load(target_path)
display(final_df.orderBy("customer_id"))

print("\n--- Summary ---")
print(f"Total records in target: {final_df.count()}")
print(f"Records processed (CDC): {changes_df.count()}")
print(f"Efficiency: {100 * changes_df.count() / new_source_df.count():.1f}% of source data processed")
print(f"New last sync timestamp: 2024-01-04 15:00:00")

In [0]:
# ============================================
# CDC EXAMPLE 2: Slowly Changing Dimension Type 2
# ============================================
# Track history of changes over time

from pyspark.sql.functions import col, current_timestamp, lit, when
from delta.tables import DeltaTable

print("="*70)
print("SCENARIO: Track customer history (SCD Type 2)")
print("="*70)

print("""
SCD Type 2 = Keep full history of changes

For each record, track:
  - effective_date: When this version became active
  - end_date: When this version expired (NULL = current)
  - is_current: Boolean flag for current version

Example:
  customer_id | name | email | effective_date | end_date | is_current
  1 | Alice | alice@old.com | 2024-01-01 | 2024-01-15 | False
  1 | Alice | alice@new.com | 2024-01-15 | NULL | True
""")

print("\n--- Step 1: Initial Data ---\n")

initial_data = [
    (1, "Alice", "alice@email.com", "active", "2024-01-01", None, True),
    (2, "Bob", "bob@email.com", "active", "2024-01-01", None, True),
]

schema = ["customer_id", "name", "email", "status", "effective_date", "end_date", "is_current"]
initial_df = spark.createDataFrame(initial_data, schema)

target_path_scd2 = "/tmp/cdc_example/customers_scd2"
initial_df.write.format("delta").mode("overwrite").save(target_path_scd2)

print("Initial SCD Type 2 table:")
display(spark.read.format("delta").load(target_path_scd2))

print("\n--- Step 2: Changes Detected ---\n")

# Alice changed her email
changes = [
    (1, "Alice", "alice.new@email.com", "active", "2024-01-15"),
]

changes_df = spark.createDataFrame(
    changes,
    ["customer_id", "name", "email", "status", "change_date"]
)

print("Changes to apply:")
display(changes_df)

print("\n--- Step 3: Apply SCD Type 2 Logic ---\n")

target_table = DeltaTable.forPath(spark, target_path_scd2)

# Step 3a: Expire old records (set end_date and is_current = False)
print("Expiring old versions...")
target_table.alias("target").merge(
    changes_df.alias("source"),
    "target.customer_id = source.customer_id AND target.is_current = True"
).whenMatchedUpdate(
    set={
        "end_date": col("source.change_date"),
        "is_current": lit(False)
    }
).execute()

# Step 3b: Insert new versions
print("Inserting new versions...")
new_versions_df = changes_df.select(
    col("customer_id"),
    col("name"),
    col("email"),
    col("status"),
    col("change_date").alias("effective_date"),
    lit(None).cast("string").alias("end_date"),
    lit(True).alias("is_current")
)

new_versions_df.write.format("delta").mode("append").save(target_path_scd2)

print("\n--- Step 4: View History ---\n")

final_df = spark.read.format("delta").load(target_path_scd2)

print("Complete history (all versions):")
display(final_df.orderBy("customer_id", "effective_date"))

print("\nCurrent snapshot (is_current = True):")
display(final_df.filter(col("is_current") == True))

print("\nAlice's history:")
display(final_df.filter(col("customer_id") == 1).orderBy("effective_date"))

print("""
✓ Old record marked as expired (end_date set, is_current = False)
✓ New record inserted (end_date = NULL, is_current = True)
✓ Full history preserved for auditing and time-travel queries
""")

In [0]:
# ============================================
# CDC EXAMPLE 3: Delta Lake Native CDC
# ============================================
# Use Delta Lake's built-in Change Data Feed

from pyspark.sql.functions import col
from delta.tables import DeltaTable

print("="*70)
print("SCENARIO: Delta Lake Change Data Feed (CDF)")
print("="*70)

print("""
Delta Lake CDF = Automatic change tracking

Features:
  • Automatic capture of all changes (INSERT, UPDATE, DELETE)
  • No manual tracking needed
  • Efficient storage
  • Query changes by version or timestamp

Change types:
  - insert: New rows
  - update_preimage: Old values before update
  - update_postimage: New values after update
  - delete: Deleted rows
""")

print("\n--- Step 1: Enable Change Data Feed ---\n")

target_path_cdf = "/tmp/cdc_example/customers_cdf"

# Create Delta table with CDF enabled
initial_data = [
    (1, "Alice", "alice@email.com", "active"),
    (2, "Bob", "bob@email.com", "active"),
    (3, "Charlie", "charlie@email.com", "inactive"),
]

initial_df = spark.createDataFrame(
    initial_data,
    ["customer_id", "name", "email", "status"]
)

# Write with CDF enabled
initial_df.write.format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .save(target_path_cdf)

print("✓ Table created with Change Data Feed enabled")
print("\nInitial data:")
display(spark.read.format("delta").load(target_path_cdf))

print("\n--- Step 2: Make Changes to Table ---\n")

target_table = DeltaTable.forPath(spark, target_path_cdf)

# Change 1: Update Bob's email
print("Change 1: Updating Bob's email...")
target_table.update(
    condition="customer_id = 2",
    set={"email": lit("bob.smith@email.com"), "name": lit("Bob Smith")}
)

# Change 2: Insert new customer
print("Change 2: Inserting Diana...")
new_customer = spark.createDataFrame(
    [(4, "Diana", "diana@email.com", "active")],
    ["customer_id", "name", "email", "status"]
)
target_table.alias("target").merge(
    new_customer.alias("source"),
    "target.customer_id = source.customer_id"
).whenNotMatchedInsertAll().execute()

# Change 3: Delete Charlie
print("Change 3: Deleting Charlie...")
target_table.delete("customer_id = 3")

print("\n--- Step 3: Query Change Data ---\n")

# Read all changes since version 0
changes_df = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 0) \
    .load(target_path_cdf)

print("All changes captured by Delta Lake CDF:")
print("Columns: _change_type, _commit_version, _commit_timestamp")
display(changes_df.orderBy("_commit_version", "_change_type"))

print("\n--- Step 4: Analyze Changes by Type ---\n")

print("INSERT operations:")
inserts = changes_df.filter(col("_change_type") == "insert")
display(inserts.select("customer_id", "name", "email", "_commit_version"))

print("\nUPDATE operations (before and after):")
updates = changes_df.filter(col("_change_type").contains("update"))
display(updates.select("customer_id", "name", "email", "_change_type", "_commit_version"))

print("\nDELETE operations:")
deletes = changes_df.filter(col("_change_type") == "delete")
display(deletes.select("customer_id", "name", "email", "_commit_version"))

print("\n--- Step 5: Incremental Processing ---\n")

print("""
Use Case: Downstream system needs to sync changes

Query only changes since last processed version:
""")

last_processed_version = 1

incremental_changes = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", last_processed_version + 1) \
    .load(target_path_cdf)

print(f"Changes since version {last_processed_version}:")
display(incremental_changes.select("customer_id", "name", "_change_type", "_commit_version"))

print("""
✓ No manual change tracking needed
✓ Efficient: Only store changes, not full snapshots
✓ Queryable: Filter by version or timestamp
✓ Downstream systems can process incrementally
""")

In [0]:
# ============================================
# CDC EXAMPLE 4: Production CDC with Delta Live Tables
# ============================================
# Real-world CDC pipeline using DLT

print("="*70)
print("PRODUCTION CDC PIPELINE ARCHITECTURE")
print("="*70)

print("""
┌─────────────────────────────────────────────────────────────────┐
│                     CDC PIPELINE FLOW                           │
└─────────────────────────────────────────────────────────────────┘

  Source Database (MySQL/Postgres/MongoDB)
           │
           │ CDC Tool (Debezium/AWS DMS/Fivetran)
           ↓
  Kafka/Kinesis (Change Events)
           │
           ↓
  ┌─────────────────────┐
  │   BRONZE LAYER      │  Raw CDC events (append-only)
  │   - Captures all    │
  │   - No transformations
  └─────────────────────┘
           │
           ↓
  ┌─────────────────────┐
  │   SILVER LAYER      │  Current state (MERGE)
  │   - Apply changes   │  
  │   - Deduplicate     │
  │   - Validate        │
  └─────────────────────┘
           │
           ↓
  ┌─────────────────────┐
  │   GOLD LAYER        │  Business aggregations
  │   - Analytics       │
  │   - Reports         │
  └─────────────────────┘
""")

print("\n" + "="*70)
print("DLT CDC PIPELINE CODE")
print("="*70)

print("""
Create a new DLT pipeline with this code:
File: /Pipelines/cdc_customer_pipeline.py
""")

print('''
import dlt
from pyspark.sql.functions import col, max, row_number, current_timestamp
from pyspark.sql.window import Window

# ============================================
# BRONZE LAYER: Raw CDC Events
# ============================================

@dlt.table(
    name="bronze_customer_cdc",
    comment="Raw CDC events from source system",
    table_properties={
        "quality": "bronze",
        "pipelines.autoOptimize.managed": "true"
    }
)
def bronze_customer_cdc():
    """Ingest raw CDC events from Kafka/Kinesis/S3"""
    return (
        spark.readStream
            .format("cloudFiles")  # or kafka, kinesis
            .option("cloudFiles.format", "json")
            .option("cloudFiles.schemaLocation", "/tmp/cdc_schema/customers")
            .load("/source/cdc/customers/")
            .select(
                col("payload.customer_id").cast("int"),
                col("payload.name"),
                col("payload.email"),
                col("payload.status"),
                col("payload.updated_at").cast("timestamp"),
                col("op").alias("operation"),  # c=create, u=update, d=delete
                col("ts_ms").cast("timestamp").alias("event_timestamp"),
                current_timestamp().alias("ingestion_timestamp")
            )
    )

# ============================================
# SILVER LAYER: Current State (Apply CDC)
# ============================================

@dlt.table(
    name="silver_customers",
    comment="Current state of customers with CDC applied",
    table_properties={
        "quality": "silver",
        "delta.enableChangeDataFeed": "true"  # Enable CDF for downstream
    }
)
@dlt.expect_or_drop("valid_customer_id", "customer_id IS NOT NULL")
@dlt.expect_or_drop("valid_email", "email IS NOT NULL AND email LIKE '%@%'")
def silver_customers():
    """Apply CDC changes to maintain current state"""
    
    # Read bronze CDC events
    bronze_df = dlt.read_stream("bronze_customer_cdc")
    
    # Deduplicate: Keep latest event per customer_id
    window_spec = Window.partitionBy("customer_id").orderBy(col("event_timestamp").desc())
    
    latest_df = (
        bronze_df
            .withColumn("row_num", row_number().over(window_spec))
            .filter(col("row_num") == 1)
            .drop("row_num")
    )
    
    # Filter out deletes (handle separately)
    upserts_df = latest_df.filter(col("operation") != "d")
    
    return upserts_df.select(
        "customer_id",
        "name",
        "email",
        "status",
        "updated_at",
        "event_timestamp"
    )

# Apply MERGE for SCD Type 1 (current state)
dlt.create_streaming_live_table(
    name="silver_customers",
    expect_all_or_drop={"valid_customer_id": "customer_id IS NOT NULL"},
)

# ============================================
# SILVER LAYER: Deleted Records Tracking
# ============================================

@dlt.table(
    name="silver_customers_deleted",
    comment="Soft-deleted customers for audit trail"
)
def silver_customers_deleted():
    """Track deleted customers separately"""
    bronze_df = dlt.read_stream("bronze_customer_cdc")
    
    return (
        bronze_df
            .filter(col("operation") == "d")
            .select(
                "customer_id",
                "name",
                "email",
                col("event_timestamp").alias("deleted_at")
            )
    )

# ============================================
# GOLD LAYER: Business Aggregations
# ============================================

@dlt.table(
    name="gold_customer_metrics",
    comment="Customer summary metrics"
)
def gold_customer_metrics():
    """Business-level customer metrics"""
    return (
        dlt.read("silver_customers")
            .groupBy("status")
            .agg(
                count("*").alias("customer_count"),
                max("updated_at").alias("last_update")
            )
    )
''')

print("\n" + "="*70)
print("DEPLOY & RUN")
print("="*70)

print("""
Step 1: Create DLT Pipeline
  • Go to Workflows > Delta Live Tables
  • Create Pipeline
  • Set notebook path: /Pipelines/cdc_customer_pipeline
  • Set target database: customer_db
  • Enable CDF: Add configuration delta.enableChangeDataFeed = true

Step 2: Configure Source
  • Set up CDC tool (Debezium/DMS)
  • Point to source database
  • Configure Kafka/Kinesis or S3 landing zone

Step 3: Run Pipeline
  • Start: Triggered or Continuous mode
  • Monitor: Built-in DLT UI
  • Query: Use silver_customers for current state

Step 4: Downstream Consumption
  • Analytics: Query gold_customer_metrics
  • ML: Use silver_customers
  • Reporting: Subscribe to CDF from silver layer
""")

print("\n" + "="*70)
print("CDC BEST PRACTICES")
print("="*70)

print("""
✓ USE DELTA LAKE
  • Built-in CDC support
  • ACID transactions
  • Time travel

✓ ENABLE CHANGE DATA FEED
  • Automatic change tracking
  • Efficient downstream processing

✓ IMPLEMENT DATA QUALITY
  • Validate primary keys
  • Check for duplicates
  • Handle late-arriving data

✓ MONITOR LATENCY
  • Track end-to-end delay
  • Alert on lag

✓ HANDLE DELETES CAREFULLY
  • Soft delete (flag) vs hard delete
  • Compliance requirements (GDPR)
  • Audit trail

✓ TEST EDGE CASES
  • Out-of-order events
  • Duplicate events
  • Schema changes
  • Large batch updates

✓ PARTITION STRATEGICALLY
  • By date for time-series
  • By region/category for large tables

✓ CHECKPOINT REGULARLY
  • Track processed events
  • Enable idempotent processing
  • Recover from failures
""")

In [0]:
%sql
-- ============================================
-- CDC EXAMPLE 5: SQL MERGE for CDC
-- ============================================
-- MERGE = Upsert operation (UPDATE + INSERT)

-- ============================================
-- SCENARIO: Daily customer updates from source
-- ============================================

-- Step 1: Create target table (current state)
CREATE OR REPLACE TABLE demo_catalog.default.customers_target (
  customer_id INT,
  name STRING,
  email STRING,
  status STRING,
  updated_at TIMESTAMP
) USING DELTA;

-- Initial data
INSERT INTO demo_catalog.default.customers_target VALUES
  (1, 'Alice', 'alice@email.com', 'active', '2024-01-01 10:00:00'),
  (2, 'Bob', 'bob@email.com', 'active', '2024-01-01 10:00:00'),
  (3, 'Charlie', 'charlie@email.com', 'inactive', '2024-01-02 11:00:00');

SELECT * FROM demo_catalog.default.customers_target;

-- Step 2: Simulate CDC changes (staging table)
CREATE OR REPLACE TEMP VIEW customers_changes AS
SELECT * FROM VALUES
  (2, 'Bob Smith', 'bob.smith@email.com', 'inactive', '2024-01-04 14:00:00'),  -- UPDATE
  (4, 'Diana', 'diana@email.com', 'active', '2024-01-04 15:00:00'),           -- INSERT
  (3, 'Charlie Brown', 'charlie.b@email.com', 'active', '2024-01-04 16:00:00') -- UPDATE
AS changes(customer_id, name, email, status, updated_at);

SELECT 'Changes to apply:' as message;
SELECT * FROM customers_changes;

-- Step 3: Apply CDC using MERGE
MERGE INTO demo_catalog.default.customers_target AS target
USING customers_changes AS source
ON target.customer_id = source.customer_id

-- When matched: UPDATE existing record
WHEN MATCHED THEN
  UPDATE SET
    target.name = source.name,
    target.email = source.email,
    target.status = source.status,
    target.updated_at = source.updated_at

-- When not matched: INSERT new record
WHEN NOT MATCHED THEN
  INSERT (customer_id, name, email, status, updated_at)
  VALUES (source.customer_id, source.name, source.email, source.status, source.updated_at);

-- Step 4: Verify results
SELECT 'Final state after MERGE:' as message;
SELECT * FROM demo_catalog.default.customers_target ORDER BY customer_id;

-- ============================================
-- ADVANCED: MERGE with DELETE (Full CDC)
-- ============================================

-- Scenario: Source sends delete flags
CREATE OR REPLACE TEMP VIEW customers_changes_with_deletes AS
SELECT * FROM VALUES
  (1, 'Alice', 'alice@email.com', 'active', '2024-01-05 10:00:00', false),
  (2, 'Bob Smith', 'bob.smith@email.com', 'inactive', '2024-01-05 11:00:00', true),  -- DELETE
  (5, 'Eve', 'eve@email.com', 'active', '2024-01-05 12:00:00', false)                -- INSERT
AS changes(customer_id, name, email, status, updated_at, is_deleted);

SELECT 'Changes with deletes:' as message;
SELECT * FROM customers_changes_with_deletes;

-- MERGE with conditional delete
MERGE INTO demo_catalog.default.customers_target AS target
USING customers_changes_with_deletes AS source
ON target.customer_id = source.customer_id

-- When matched and deleted: DELETE
WHEN MATCHED AND source.is_deleted = true THEN
  DELETE

-- When matched and not deleted: UPDATE
WHEN MATCHED AND source.is_deleted = false THEN
  UPDATE SET
    target.name = source.name,
    target.email = source.email,
    target.status = source.status,
    target.updated_at = source.updated_at

-- When not matched and not deleted: INSERT
WHEN NOT MATCHED AND source.is_deleted = false THEN
  INSERT (customer_id, name, email, status, updated_at)
  VALUES (source.customer_id, source.name, source.email, source.status, source.updated_at);

SELECT 'Final state after MERGE with DELETE:' as message;
SELECT * FROM demo_catalog.default.customers_target ORDER BY customer_id;

-- ============================================
-- PERFORMANCE TIPS
-- ============================================

-- Tip 1: Filter source data before MERGE
-- Only merge records that actually changed

-- Tip 2: Use MERGE with INSERT ONLY when appropriate
-- If you know there are no updates, use INSERT ONLY
MERGE INTO demo_catalog.default.customers_target AS target
USING customers_changes AS source
ON target.customer_id = source.customer_id
WHEN NOT MATCHED THEN
  INSERT *;

-- Tip 3: Partition target table for better performance
-- ALTER TABLE customers_target
-- SET TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true');

-- Tip 4: Monitor MERGE performance
DESCRIBE HISTORY demo_catalog.default.customers_target LIMIT 5;

In [0]:
# ============================================
# CREATE DATABRICKS JOBS - Python Code
# ============================================
# Three methods: UI, Databricks CLI, or Python API

print("="*70)
print("METHOD 1: CREATE JOB VIA UI (Manual)")
print("="*70)

print("""
1. Go to Workflows > Jobs
2. Click 'Create Job'
3. Configure:
   - Job name
   - Tasks (notebook, Python, SQL, etc.)
   - Schedule (cron expression)
   - Cluster settings
   - Notifications
4. Click 'Create' and 'Run Now'
""")

print("="*70)
print("METHOD 2: PYTHON API (Programmatic)")
print("="*70)

print("\n--- Install Databricks SDK ---\n")
print("""
%pip install databricks-sdk
import dbutils
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

# Initialize client (uses current auth)
w = WorkspaceClient()
""")

print("\n--- Example 1: Simple Notebook Job ---\n")
print("""
# Create a job that runs a notebook daily at 2 AM
job = w.jobs.create(
    name="Daily ETL Pipeline",
    tasks=[
        jobs.Task(
            task_key="etl_task",
            description="Run ETL notebook",
            notebook_task=jobs.NotebookTask(
                notebook_path="/Users/user@company.com/ETL_Pipeline",
                base_parameters={"date": "{{job.start_time.iso_date}}"}
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=2
            )
        )
    ],
    schedule=jobs.CronSchedule(
        quartz_cron_expression="0 0 2 * * ?",  # 2 AM daily
        timezone_id="America/Los_Angeles"
    ),
    email_notifications=jobs.JobEmailNotifications(
        on_success=["team@company.com"],
        on_failure=["oncall@company.com"]
    )
)

print(f"Job created with ID: {job.job_id}")
""")

print("\n--- Example 2: Multi-Task Job with Dependencies ---\n")
print("""
# Bronze -> Silver -> Gold pipeline with dependencies
job = w.jobs.create(
    name="Multi-Layer Data Pipeline",
    tasks=[
        # Task 1: Bronze (ingest raw data)
        jobs.Task(
            task_key="bronze_ingest",
            description="Ingest raw data",
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipelines/bronze_ingest"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=2
            )
        ),
        # Task 2: Silver (clean data) - depends on bronze
        jobs.Task(
            task_key="silver_clean",
            description="Clean and validate",
            depends_on=[jobs.TaskDependency(task_key="bronze_ingest")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipelines/silver_clean"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=2
            )
        ),
        # Task 3: Gold (aggregate) - depends on silver
        jobs.Task(
            task_key="gold_aggregate",
            description="Business aggregations",
            depends_on=[jobs.TaskDependency(task_key="silver_clean")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Pipelines/gold_aggregate"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=1
            )
        ),
        # Task 4: Send Report - depends on gold
        jobs.Task(
            task_key="send_report",
            description="Email daily report",
            depends_on=[jobs.TaskDependency(task_key="gold_aggregate")],
            notebook_task=jobs.NotebookTask(
                notebook_path="/Reporting/daily_report"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=1
            )
        )
    ],
    schedule=jobs.CronSchedule(
        quartz_cron_expression="0 0 3 * * ?",  # 3 AM daily
        timezone_id="UTC"
    )
)

print(f"Multi-task job created: {job.job_id}")
""")

print("\n--- Example 3: Python Script Task ---\n")
print("""
# Run a Python file instead of notebook
job = w.jobs.create(
    name="ML Model Training",
    tasks=[
        jobs.Task(
            task_key="train_model",
            python_wheel_task=jobs.PythonWheelTask(
                package_name="ml_training",
                entry_point="train",
                parameters=["--model", "xgboost", "--epochs", "100"]
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="g4dn.xlarge",  # GPU instance
                num_workers=0  # Single node for ML
            ),
            libraries=[
                jobs.Library(pypi=jobs.PythonPyPiLibrary(package="xgboost==2.0.0")),
                jobs.Library(pypi=jobs.PythonPyPiLibrary(package="mlflow==2.10.0"))
            ]
        )
    ]
)
""")

print("\n--- Example 4: DLT Pipeline Task ---\n")
print("""
# Trigger a DLT pipeline from a job
job = w.jobs.create(
    name="DLT Pipeline Refresh",
    tasks=[
        jobs.Task(
            task_key="refresh_dlt",
            pipeline_task=jobs.PipelineTask(
                pipeline_id="<your-dlt-pipeline-id>",
                full_refresh=False  # Incremental refresh
            )
        )
    ],
    schedule=jobs.CronSchedule(
        quartz_cron_expression="0 0 * * * ?",  # Every hour
        timezone_id="UTC"
    )
)
""")

print("\n--- Example 5: SQL Task ---\n")
print("""
# Run SQL queries against SQL Warehouse
job = w.jobs.create(
    name="SQL Analytics Refresh",
    tasks=[
        jobs.Task(
            task_key="refresh_dashboard_data",
            sql_task=jobs.SqlTask(
                warehouse_id="<sql-warehouse-id>",
                query=jobs.SqlTaskQuery(
                    query_id="<query-id>"  # Saved query
                )
            )
        )
    ]
)
""")

print("\n--- Example 6: File Arrival Trigger ---\n")
print("""
# Trigger job when new files arrive
job = w.jobs.create(
    name="Process New Files",
    tasks=[
        jobs.Task(
            task_key="process_files",
            notebook_task=jobs.NotebookTask(
                notebook_path="/Processing/process_new_files"
            ),
            new_cluster=jobs.ClusterSpec(
                spark_version="15.4.x-scala2.12",
                node_type_id="i3.xlarge",
                num_workers=2
            )
        )
    ],
    trigger=jobs.TriggerSettings(
        file_arrival=jobs.FileArrivalTriggerConfiguration(
            url="s3://my-bucket/incoming/",
            min_time_between_triggers_seconds=60
        )
    )
)
""")

print("\n" + "="*70)
print("JOB OPERATIONS")
print("="*70)

print("""
# Run a job
run = w.jobs.run_now(job_id=123)
print(f"Run started: {run.run_id}")

# Get job status
run_status = w.jobs.get_run(run_id=run.run_id)
print(f"Status: {run_status.state.life_cycle_state}")

# List all jobs
jobs_list = w.jobs.list()
for job in jobs_list:
    print(f"Job: {job.settings.name} (ID: {job.job_id})")

# Delete a job
w.jobs.delete(job_id=123)

# Update job schedule
w.jobs.update(
    job_id=123,
    new_settings=jobs.JobSettings(
        schedule=jobs.CronSchedule(
            quartz_cron_expression="0 0 4 * * ?",  # Change to 4 AM
            timezone_id="UTC"
        )
    )
)
""")

print("\n" + "="*70)
print("CRON EXPRESSION EXAMPLES")
print("="*70)

print("""
Format: seconds minutes hours day month day-of-week year

Daily at 2 AM:        0 0 2 * * ? *
Every hour:           0 0 * * * ? *
Every 15 minutes:     0 */15 * * * ? *
Every Monday 8 AM:    0 0 8 ? * MON *
First day of month:   0 0 0 1 * ? *
Weekdays at noon:     0 0 12 ? * MON-FRI *
""")

print("\n" + "="*70)
print("BEST PRACTICES")
print("="*70)

print("""
✓ Use job clusters (not all-purpose) for production
✓ Set appropriate retry policies
✓ Configure email/Slack alerts
✓ Use task dependencies for complex workflows
✓ Pass parameters for reusable notebooks
✓ Monitor job run history
✓ Set max concurrent runs to prevent overlaps
✓ Use version-controlled notebooks/scripts
✓ Tag jobs for cost tracking
""")

In [0]:
# ============================================
# COMPLETE DLT PIPELINE WITH EXPECTATIONS
# ============================================
# This is a production-ready example showing:
# - Bronze/Silver/Gold architecture
# - Comprehensive expectations at each layer
# - Streaming and batch tables
# - Real-world data quality checks

print("="*80)
print("COMPLETE END-TO-END DLT PIPELINE")
print("="*80)

print("""
Scenario: E-commerce platform processing customer orders

Pipeline Flow:
  Raw S3 Data -> Bronze -> Silver -> Gold -> Business Reports
  
Data Quality Strategy:
  Bronze: Track issues (warn)
  Silver: Clean data (drop bad rows)
  Gold:   Enforce critical rules (fail on violation)
""")

print("\n" + "="*80)
print("STEP 1: CREATE DLT PIPELINE NOTEBOOK")
print("="*80)

print("""
Save this code in a notebook, then create a DLT pipeline in UI:
1. Go to Workflows -> Delta Live Tables
2. Click 'Create Pipeline'
3. Select your notebook
4. Configure settings
5. Click 'Start'

Full Pipeline Code:
""")

print("""
# ============================================
# File: dlt_ecommerce_pipeline.py
# ============================================

import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

# ============================================
# BRONZE LAYER: Ingest Raw Data
# ============================================

@dlt.table(
    name="bronze_orders",
    comment="Raw orders from S3 with Auto Loader",
    table_properties={
        "quality": "bronze",
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect("has_order_id", "order_id IS NOT NULL")
@dlt.expect("has_timestamp", "order_timestamp IS NOT NULL")
@dlt.expect("has_customer", "customer_id IS NOT NULL")
def bronze_orders():
    \"\"\"Ingest raw JSON orders from S3 using Auto Loader\"\"\"
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.schemaLocation", "/mnt/schemas/orders")
            .option("cloudFiles.inferColumnTypes", "true")
            .load("s3://my-bucket/raw-orders/")
            .withColumn("ingestion_time", current_timestamp())
    )


@dlt.table(
    name="bronze_customers",
    comment="Raw customer data from database CDC"
)
@dlt.expect("has_customer_id", "customer_id IS NOT NULL")
def bronze_customers():
    \"\"\"Ingest customer data via Change Data Capture\"\"\"
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "parquet")
            .load("s3://my-bucket/cdc-customers/")
    )


@dlt.table(
    name="bronze_products",
    comment="Product catalog - batch table"
)
def bronze_products():
    \"\"\"Load product catalog (batch, updated daily)\"\"\"
    return (
        spark.read
            .format("delta")
            .load("/mnt/warehouse/product_catalog")
    )


# ============================================
# SILVER LAYER: Clean and Validate
# ============================================

@dlt.table(
    name="silver_orders",
    comment="Clean orders with quality checks applied",
    table_properties={
        "quality": "silver"
    }
)
# Drop invalid orders
@dlt.expect_or_drop("valid_email", "customer_email RLIKE '^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\\\.[a-zA-Z]{2,}$'")
@dlt.expect_or_drop("positive_amount", "order_total > 0")
@dlt.expect_or_drop("reasonable_amount", "order_total < 50000")
@dlt.expect_or_drop("valid_status", "order_status IN ('pending', 'processing', 'shipped', 'delivered', 'cancelled', 'refunded')")
@dlt.expect_or_drop("valid_payment", "payment_method IN ('credit_card', 'debit_card', 'paypal', 'bank_transfer', 'crypto')")
@dlt.expect_or_drop("future_check", "order_timestamp <= current_timestamp()")
@dlt.expect_or_drop("recent_order", "order_timestamp >= '2020-01-01'")

# Critical: Fail if customer_id missing
@dlt.expect_or_fail("has_customer_ref", "customer_id IS NOT NULL")

def silver_orders():
    \"\"\"Clean orders with comprehensive validation\"\"\"
    return (
        dlt.read_stream("bronze_orders")
            # Add derived columns
            .withColumn("order_date", to_date(col("order_timestamp")))
            .withColumn("order_year", year(col("order_timestamp")))
            .withColumn("order_month", month(col("order_timestamp")))
            .withColumn("order_hour", hour(col("order_timestamp")))
            # Calculate order metrics
            .withColumn("item_count", size(col("items")))  # JSON array
            .withColumn("discount_applied", col("discount_amount") > 0)
    )


@dlt.table(
    name="silver_customers",
    comment="Enriched customer profiles"
)
@dlt.expect_or_drop("valid_email", "email IS NOT NULL AND email LIKE '%@%'")
@dlt.expect_or_drop("valid_phone", "phone IS NULL OR LENGTH(phone) >= 10")
@dlt.expect_or_drop("valid_age", "age IS NULL OR (age >= 18 AND age <= 120)")
@dlt.expect_or_drop("valid_country", "country_code IN ('US', 'CA', 'UK', 'AU', 'DE', 'FR', 'JP')")
def silver_customers():
    \"\"\"Deduplicated and validated customer data\"\"\"
    return (
        dlt.read_stream("bronze_customers")
            # Deduplicate by taking latest record
            .withWatermark("updated_at", "1 day")
            .dropDuplicates(["customer_id"])
    )


# ============================================
# GOLD LAYER: Business Aggregates
# ============================================

@dlt.table(
    name="gold_daily_revenue",
    comment="Daily revenue metrics for executive dashboard",
    table_properties={
        "quality": "gold"
    }
)
# Critical business metrics - must be perfect
@dlt.expect_or_fail("revenue_not_null", "total_revenue IS NOT NULL")
@dlt.expect_or_fail("revenue_positive", "total_revenue > 0")
@dlt.expect_or_fail("order_count_positive", "order_count > 0")
@dlt.expect_or_fail("reasonable_revenue", "total_revenue < 10000000")  # Sanity check
@dlt.expect_or_fail("reasonable_orders", "order_count < 100000")  # Sanity check
def gold_daily_revenue():
    \"\"\"Daily revenue aggregations for reporting\"\"\"
    return (
        dlt.read("silver_orders")
            .filter(col("order_status").isin(["delivered", "shipped"]))
            .groupBy("order_date")
            .agg(
                sum("order_total").alias("total_revenue"),
                count("order_id").alias("order_count"),
                avg("order_total").alias("avg_order_value"),
                countDistinct("customer_id").alias("unique_customers"),
                sum("discount_amount").alias("total_discounts")
            )
    )


@dlt.table(
    name="gold_customer_360",
    comment="Complete customer profile with purchase history"
)
@dlt.expect_or_fail("has_customer", "customer_id IS NOT NULL")
@dlt.expect_or_fail("has_metrics", "lifetime_value IS NOT NULL AND order_count IS NOT NULL")
def gold_customer_360():
    \"\"\"Customer 360 view joining orders and profile\"\"\"
    
    # Calculate customer metrics from orders
    customer_metrics = (
        dlt.read("silver_orders")
            .filter(col("order_status") == "delivered")
            .groupBy("customer_id")
            .agg(
                count("order_id").alias("order_count"),
                sum("order_total").alias("lifetime_value"),
                avg("order_total").alias("avg_order_value"),
                min("order_timestamp").alias("first_order_date"),
                max("order_timestamp").alias("last_order_date"),
                countDistinct("product_id").alias("unique_products_purchased")
            )
    )
    
    # Join with customer profile
    return (
        dlt.read("silver_customers")
            .join(customer_metrics, "customer_id", "left")
            .withColumn(
                "customer_segment",
                when(col("lifetime_value") > 5000, "VIP")
                .when(col("lifetime_value") > 1000, "Premium")
                .when(col("lifetime_value") > 100, "Regular")
                .otherwise("New")
            )
    )


@dlt.table(
    name="gold_product_performance",
    comment="Product sales analytics"
)
@dlt.expect_or_fail("has_product", "product_id IS NOT NULL")
def gold_product_performance():
    \"\"\"Product-level sales metrics\"\"\"
    
    # Explode items array from orders
    orders_items = (
        dlt.read("silver_orders")
            .filter(col("order_status") == "delivered")
            .select(
                col("order_id"),
                col("order_date"),
                explode(col("items")).alias("item")
            )
            .select(
                col("order_id"),
                col("order_date"),
                col("item.product_id").alias("product_id"),
                col("item.quantity").alias("quantity"),
                col("item.price").alias("price")
            )
    )
    
    # Join with product catalog and aggregate
    return (
        orders_items
            .join(dlt.read("bronze_products"), "product_id", "left")
            .groupBy("product_id", "product_name", "category")
            .agg(
                sum("quantity").alias("units_sold"),
                sum(col("quantity") * col("price")).alias("revenue"),
                count("order_id").alias("order_count"),
                avg("price").alias("avg_price")
            )
    )


# ============================================
# QUARANTINE TABLE: Capture Dropped Rows
# ============================================

@dlt.table(
    name="quarantine_failed_orders",
    comment="Orders that failed silver layer quality checks"
)
def quarantine_failed_orders():
    \"\"\"Capture orders dropped by silver layer for investigation\"\"\"
    # Read bronze, identify what would be dropped by silver expectations
    bronze = dlt.read_stream("bronze_orders")
    
    return bronze.filter(
        # Negation of silver expectations
        ~(col("customer_email").rlike("^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\\\.[a-zA-Z]{2,}$")) |
        ~(col("order_total") > 0) |
        ~(col("order_total") < 50000) |
        col("customer_id").isNull()
    ).withColumn("quarantine_reason", 
        when(~col("customer_email").rlike("^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\\\.[a-zA-Z]{2,}$"), "Invalid email")
        .when(~(col("order_total") > 0), "Non-positive amount")
        .when(~(col("order_total") < 50000), "Amount too high")
        .when(col("customer_id").isNull(), "Missing customer_id")
        .otherwise("Multiple issues")
    )
""")

print("\n" + "="*80)
print("STEP 2: CREATE THE PIPELINE IN UI")
print("="*80)

print("""
1. Navigate to Workflows -> Delta Live Tables
2. Click 'Create Pipeline'
3. Configure:
   - Pipeline name: "ecommerce_data_pipeline"
   - Notebook: Select the notebook with above code
   - Target schema: "main.ecommerce"
   - Storage location: "s3://my-bucket/dlt-storage/"
   - Pipeline mode: Triggered or Continuous
   - Cluster mode: Enhanced autoscaling
4. Click 'Create'
5. Click 'Start'

Pipeline will:
  ✓ Auto-detect table dependencies
  ✓ Create execution graph
  ✓ Run bronze -> silver -> gold
  ✓ Track all expectation violations
  ✓ Display metrics in UI
""")

print("\n" + "="*80)
print("STEP 3: MONITOR DATA QUALITY")
print("="*80)

print("""
After pipeline runs, view metrics in DLT UI:

1. Data Quality Tab:
   - Expectation success rates
   - Violated row counts
   - Historical trends

2. Lineage Tab:
   - Visual dependency graph
   - Bronze -> Silver -> Gold flow
   - Click any table to see details

3. Event Log:
   - Query violation details
   
SELECT 
  dataset_name,
  expectation_name,
  SUM(passed_records) as passed,
  SUM(failed_records) as failed
FROM event_log('main.ecommerce.ecommerce_data_pipeline_events')
WHERE event_type = 'flow_progress'
GROUP BY dataset_name, expectation_name;
""")

print("\n" + "="*80)
print("PIPELINE BENEFITS")
print("="*80)

print("""
✓ Data Quality: Automatic validation at every layer
✓ Lineage: Track data flow from source to report
✓ Observability: Built-in metrics and monitoring
✓ Recovery: Auto-retry on failure
✓ Scalability: Auto-scaling clusters
✓ Simplicity: Declarative code, complex orchestration
✓ Cost: Only pay when pipeline runs
""")

print("\n" + "="*80)
print("END OF COMPLETE DLT PIPELINE EXAMPLE")
print("="*80)

In [0]:
# ============================================
# DELTA LIVE TABLES - EXPECTATIONS
# ============================================
# Expectations are data quality constraints that validate your data
# They are the KILLER FEATURE of DLT

print("="*80)
print("WHAT ARE DLT EXPECTATIONS?")
print("="*80)

print("""
Expectations are data quality rules you define for your tables.
DLT automatically:
  ✓ Validates every row against expectations
  ✓ Tracks violations in metrics
  ✓ Takes action (warn, drop, or fail)
  ✓ Reports quality metrics in UI

Think of them as:
  • Assert statements for your data
  • Unit tests that run on every record
  • Data quality gates in your pipeline
""")

print("\n" + "="*80)
print("THREE TYPES OF EXPECTATIONS")
print("="*80)

print("""
================================================================
 Type 1: @expect - WARN ONLY
================================================================
   • Logs violations but continues processing
   • All rows pass through (even invalid ones)
   • Use for: Monitoring, soft checks
   • Example: Track completeness but don't block

================================================================
 Type 2: @expect_or_drop - QUARANTINE BAD ROWS
================================================================
   • Drops rows that violate expectations
   • Valid rows continue, invalid rows excluded
   • Use for: Data cleansing, filtering bad data
   • Example: Drop records with invalid emails

================================================================
 Type 3: @expect_or_fail - STOP PIPELINE
================================================================
   • Pipeline FAILS if ANY row violates
   • No data processed if expectations fail
   • Use for: Critical business rules
   • Example: Revenue must be positive
""")

print("\n" + "="*80)
print("PYTHON API - COMPLETE EXAMPLES")
print("="*80)

print("""
# NOTE: This code runs ONLY in DLT pipelines, not regular notebooks
# Save this in a DLT pipeline notebook

import dlt
from pyspark.sql.functions import col

# ============================================
# Example 1: BRONZE LAYER - Warn Only
# ============================================

@dlt.table(
    name="bronze_orders",
    comment="Raw orders with quality tracking"
)
@dlt.expect("valid_order_id", "order_id IS NOT NULL")
@dlt.expect("positive_amount", "amount > 0")
@dlt.expect("valid_timestamp", "order_timestamp IS NOT NULL")
def bronze_orders():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .load("/mnt/raw/orders/")
    )

# Result: All rows pass through, violations are LOGGED
# Use Case: Track data quality without blocking pipeline


# ============================================
# Example 2: SILVER LAYER - Drop Invalid Rows
# ============================================

@dlt.table(
    name="silver_orders_clean",
    comment="Clean orders with invalid rows dropped"
)
@dlt.expect_or_drop("valid_email", "email IS NOT NULL AND email LIKE '%@%'")
@dlt.expect_or_drop("valid_amount", "amount >= 0 AND amount < 1000000")
@dlt.expect_or_drop("valid_status", "status IN ('pending', 'completed', 'cancelled')")
@dlt.expect_or_drop("valid_date", "order_date >= '2020-01-01'")
def silver_orders_clean():
    return dlt.read_stream("bronze_orders")

# Result: Only valid rows pass through, bad rows DROPPED
# Dropped rows are tracked in metrics
# Use Case: Ensure downstream tables have clean data


# ============================================
# Example 3: GOLD LAYER - Fail Pipeline
# ============================================

@dlt.table(
    name="gold_daily_revenue",
    comment="Critical business metrics - must be perfect"
)
@dlt.expect_or_fail("revenue_not_null", "total_revenue IS NOT NULL")
@dlt.expect_or_fail("revenue_positive", "total_revenue > 0")
@dlt.expect_or_fail("reasonable_revenue", "total_revenue < 10000000")  # Sanity check
def gold_daily_revenue():
    return (
        dlt.read("silver_orders_clean")
            .groupBy("order_date")
            .agg({"amount": "sum"})
            .withColumnRenamed("sum(amount)", "total_revenue")
    )

# Result: Pipeline FAILS if any expectation violated
# Use Case: Critical metrics that must be 100% correct


# ============================================
# Example 4: Multiple Expectations on One Table
# ============================================

@dlt.table(
    name="silver_customers",
    comment="Customer data with comprehensive quality checks"
)
# Warn only - track completeness
@dlt.expect("has_phone", "phone IS NOT NULL")
@dlt.expect("has_address", "address IS NOT NULL")

# Drop invalid - enforce data quality
@dlt.expect_or_drop("valid_email", "email IS NOT NULL AND email RLIKE '^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\\\.[a-zA-Z]{2,}$'")
@dlt.expect_or_drop("valid_age", "age >= 18 AND age <= 120")
@dlt.expect_or_drop("valid_country", "country_code IN ('US', 'CA', 'UK', 'AU')")

# Fail pipeline - critical business rules
@dlt.expect_or_fail("unique_customer_id", "customer_id IS NOT NULL")
def silver_customers():
    return dlt.read("bronze_customers")

# Result: Mix of warn, drop, and fail behaviors
# Flexible data quality enforcement


# ============================================
# Example 5: Complex Expectations with SQL
# ============================================

@dlt.table(
    name="silver_transactions",
    comment="Financial transactions with strict validation"
)
@dlt.expect_or_drop(
    "valid_transaction",
    \"\"\"transaction_amount > 0 
       AND transaction_date >= '2020-01-01' 
       AND transaction_date <= CURRENT_DATE()
       AND account_id IS NOT NULL
       AND account_id != ''\"\"\"
)
@dlt.expect_or_fail(
    "balanced_transaction",
    "debit_amount = credit_amount"  # Double-entry bookkeeping
)
def silver_transactions():
    return dlt.read("bronze_transactions")

# Result: Complex multi-condition validation


# ============================================
# Example 6: Expectations with Named Constraints
# ============================================

@dlt.table(
    name="silver_products",
    comment="Product catalog with quality expectations"
)
@dlt.expect_all({
    "valid_sku": "sku IS NOT NULL AND LENGTH(sku) = 8",
    "valid_price": "price > 0 AND price < 100000",
    "valid_category": "category IN ('Electronics', 'Clothing', 'Food')",
    "valid_stock": "stock_quantity >= 0"
})
def silver_products():
    return dlt.read("bronze_products")

# expect_all: Apply multiple expectations at once
# Named constraints make metrics easier to track
""")

print("\n" + "="*80)
print("SQL API - EXAMPLES")
print("="*80)

print("""
-- ============================================
-- SQL Syntax for DLT Expectations
-- ============================================

-- Example 1: Bronze with warnings
CREATE OR REFRESH STREAMING LIVE TABLE bronze_events (
  CONSTRAINT valid_event_id EXPECT (event_id IS NOT NULL),
  CONSTRAINT valid_timestamp EXPECT (event_time IS NOT NULL)
)
COMMENT "Raw events with quality tracking"
AS SELECT * FROM cloud_files(
  "/mnt/events/",
  "json"
);


-- Example 2: Silver with row dropping
CREATE OR REFRESH STREAMING LIVE TABLE silver_events_clean (
  CONSTRAINT valid_user EXPECT (user_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_event_type EXPECT (event_type IN ('click', 'view', 'purchase')) ON VIOLATION DROP ROW,
  CONSTRAINT recent_event EXPECT (event_time > '2023-01-01') ON VIOLATION DROP ROW
)
COMMENT "Clean events with invalid rows removed"
AS SELECT * FROM STREAM(LIVE.bronze_events);


-- Example 3: Gold with pipeline failure
CREATE OR REFRESH LIVE TABLE gold_daily_metrics (
  CONSTRAINT revenue_exists EXPECT (total_revenue IS NOT NULL) ON VIOLATION FAIL UPDATE,
  CONSTRAINT positive_revenue EXPECT (total_revenue > 0) ON VIOLATION FAIL UPDATE
)
COMMENT "Critical business metrics"
AS 
SELECT 
  DATE(event_time) as event_date,
  COUNT(*) as event_count,
  SUM(revenue) as total_revenue
FROM LIVE.silver_events_clean
WHERE event_type = 'purchase'
GROUP BY DATE(event_time);
""")

print("\n" + "="*80)
print("REAL-WORLD EXAMPLE: E-COMMERCE PIPELINE")
print("="*80)

print("""
import dlt
from pyspark.sql.functions import *

# ============================================
# Bronze: Ingest raw data with tracking
# ============================================

@dlt.table(comment="Raw orders from S3")
@dlt.expect("has_order_id", "order_id IS NOT NULL")
@dlt.expect("has_timestamp", "created_at IS NOT NULL")
def bronze_orders():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .load("s3://orders-bucket/raw/")
    )

# Tracks: How many orders lack IDs or timestamps?


# ============================================
# Silver: Clean and validate
# ============================================

@dlt.table(comment="Clean orders ready for analytics")
# Drop invalid data
@dlt.expect_or_drop("valid_email", "customer_email LIKE '%@%'")
@dlt.expect_or_drop("positive_total", "order_total > 0")
@dlt.expect_or_drop("valid_status", "order_status IN ('pending', 'shipped', 'delivered', 'cancelled')")
@dlt.expect_or_drop("valid_payment", "payment_method IN ('credit_card', 'paypal', 'bank_transfer')")
@dlt.expect_or_drop("reasonable_total", "order_total < 50000")  # Fraud detection

# Critical: Must not be null
@dlt.expect_or_fail("has_customer_id", "customer_id IS NOT NULL")
def silver_orders():
    return (
        dlt.read_stream("bronze_orders")
            .withColumn("order_date", to_date(col("created_at")))
    )

# Result:
# - Invalid emails/amounts: DROPPED
# - Missing customer_id: PIPELINE FAILS
# - All downstream data is clean


# ============================================
# Gold: Business metrics with strict checks
# ============================================

@dlt.table(comment="Daily sales metrics for executive dashboard")
@dlt.expect_or_fail("revenue_exists", "daily_revenue IS NOT NULL")
@dlt.expect_or_fail("positive_revenue", "daily_revenue > 0")
@dlt.expect_or_fail("reasonable_orders", "order_count > 0 AND order_count < 1000000")
def gold_daily_sales():
    return (
        dlt.read("silver_orders")
            .filter(col("order_status") == "delivered")
            .groupBy("order_date")
            .agg(
                sum("order_total").alias("daily_revenue"),
                count("order_id").alias("order_count"),
                avg("order_total").alias("avg_order_value")
            )
    )

# Result: Pipeline fails if metrics are invalid
# Executives always see correct data
""")

print("\n" + "="*80)
print("MONITORING EXPECTATIONS")
print("="*80)

print("""
DLT automatically tracks all expectation violations:

1. DLT UI Dashboard:
   - View expectation metrics per table
   - See violation counts and percentages
   - Historical trends over time
   - Failed/passed runs

2. Event Log:
   - Detailed violation records
   - Which rows failed which expectations
   - Queryable via SQL

3. Metrics API:
   - Programmatic access to quality metrics
   - Integrate with monitoring tools
   - Alert on quality degradation

Example Query for Violations:

SELECT 
  flow_name,
  dataset_name,
  expectation_name,
  SUM(passed_records) as passed,
  SUM(failed_records) as failed,
  ROUND(SUM(failed_records) * 100.0 / (SUM(passed_records) + SUM(failed_records)), 2) as failure_rate_pct
FROM event_log('main.my_pipeline_events')
WHERE event_type = 'flow_progress'
GROUP BY flow_name, dataset_name, expectation_name
ORDER BY failure_rate_pct DESC;
""")

print("\n" + "="*80)
print("BEST PRACTICES")
print("="*80)

print("""
✓ BRONZE LAYER:
  • Use @expect (warn only)
  • Track data quality issues
  • Don't block ingestion
  • Log everything for investigation

✓ SILVER LAYER:
  • Use @expect_or_drop
  • Remove invalid/corrupt data
  • Ensure downstream cleanliness
  • Most expectations go here

✓ GOLD LAYER:
  • Use @expect_or_fail
  • Protect critical business metrics
  • Only for mission-critical rules
  • Executive-facing data must be perfect

✓ NAMING:
  • Use descriptive expectation names
  • Good: "valid_email_format"
  • Bad: "check1"

✓ COMPLEXITY:
  • Keep expectations simple and readable
  • One rule per expectation when possible
  • Use SQL expressions everyone understands

✓ MONITORING:
  • Set up alerts on violation rates
  • Review DLT metrics dashboard regularly
  • Investigate sudden quality changes
""")

print("\n" + "="*80)
print("END OF DLT EXPECTATIONS GUIDE")
print("="*80)

In [0]:
# ============================================
# JOB MANAGEMENT - Operations
# ============================================
# How to interact with existing jobs

print("="*70)
print("JOB MANAGEMENT OPERATIONS")
print("="*70)

print("\n--- List All Jobs ---\n")
print("""
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# List all jobs
print("All jobs in workspace:")
for job in w.jobs.list():
    print(f"  Job ID: {job.job_id}")
    print(f"  Name: {job.settings.name}")
    print(f"  Created: {job.created_time}")
    if job.settings.schedule:
        print(f"  Schedule: {job.settings.schedule.quartz_cron_expression}")
    print()
""")

print("\n--- Get Job Details ---\n")
print("""
# Get specific job info
job_id = 123
job = w.jobs.get(job_id=job_id)

print(f"Job Name: {job.settings.name}")
print(f"Creator: {job.creator_user_name}")
print(f"Number of tasks: {len(job.settings.tasks)}")
print("\nTasks:")
for task in job.settings.tasks:
    print(f"  - {task.task_key}: {task.description}")
""")

print("\n--- Run Job Manually ---\n")
print("""
# Trigger a job run
run = w.jobs.run_now(
    job_id=123,
    notebook_params={"date": "2026-03-29"}  # Pass parameters
)

print(f"Run started: {run.run_id}")
print(f"Run page: {run.run_page_url}")
""")

print("\n--- Monitor Job Run ---\n")
print("""
import time

# Poll for job completion
run_id = run.run_id
while True:
    run_status = w.jobs.get_run(run_id=run_id)
    state = run_status.state.life_cycle_state
    
    print(f"Current state: {state}")
    
    if state in ['TERMINATED', 'SKIPPED', 'INTERNAL_ERROR']:
        result = run_status.state.result_state
        print(f"Final result: {result}")
        
        if result == 'SUCCESS':
            print("✓ Job completed successfully")
        else:
            print("✗ Job failed")
            print(f"Error: {run_status.state.state_message}")
        break
    
    time.sleep(30)  # Check every 30 seconds
""")

print("\n--- Get Run History ---\n")
print("""
# List recent runs for a job
job_id = 123
runs = w.jobs.list_runs(
    job_id=job_id,
    limit=10,
    expand_tasks=True
)

print(f"Recent runs for job {job_id}:")
for run in runs:
    print(f"\n  Run ID: {run.run_id}")
    print(f"  Start: {run.start_time}")
    print(f"  State: {run.state.life_cycle_state}")
    if run.state.result_state:
        print(f"  Result: {run.state.result_state}")
    print(f"  Duration: {run.execution_duration}ms")
""")

print("\n--- Cancel Running Job ---\n")
print("""
# Cancel a running job
run_id = 456
w.jobs.cancel_run(run_id=run_id)
print(f"Job run {run_id} cancelled")
""")

print("\n--- Update Job Configuration ---\n")
print("""
from databricks.sdk.service import jobs

# Update job schedule
w.jobs.update(
    job_id=123,
    new_settings=jobs.JobSettings(
        name="Updated Job Name",
        schedule=jobs.CronSchedule(
            quartz_cron_expression="0 0 6 * * ? *",  # Change to 6 AM
            timezone_id="America/New_York"
        ),
        max_concurrent_runs=1,
        timeout_seconds=7200  # 2 hour timeout
    )
)

print("Job updated successfully")
""")

print("\n--- Delete Job ---\n")
print("""
# Delete a job (use with caution!)
job_id = 123
w.jobs.delete(job_id=job_id)
print(f"Job {job_id} deleted")
""")

print("\n" + "="*70)
print("JOB RUN STATES")
print("="*70)

print("""
Life Cycle States:
  • PENDING       - Waiting to start
  • RUNNING       - Currently executing
  • TERMINATING   - Shutting down
  • TERMINATED    - Completed (check result_state)
  • SKIPPED       - Skipped due to conditions
  • INTERNAL_ERROR - System error

Result States (when TERMINATED):
  • SUCCESS       - Completed successfully ✓
  • FAILED        - Execution failed ✗
  • TIMEDOUT      - Exceeded timeout
  • CANCELED      - Manually cancelled
""")

print("\n" + "="*70)
print("MONITORING BEST PRACTICES")
print("="*70)

print("""
✓ Set up email/Slack notifications
✓ Monitor job run duration trends
✓ Set appropriate timeouts
✓ Use max_concurrent_runs to prevent overlaps
✓ Review failed runs regularly
✓ Check cluster sizing for performance
✓ Monitor cost per job run
✓ Use tags for organization
""")

print("\n" + "="*70)
print("END OF JOB MANAGEMENT GUIDE")
print("="*70)

In [0]:
# ============================================
# DATABRICKS JOBS - Overview
# ============================================
# Jobs are scheduled workflows that run notebooks, JARs, Python scripts, or DLT pipelines

print("="*70)
print("WHAT ARE DATABRICKS JOBS?")
print("="*70)

print("\nJobs are scheduled workflows that automate data processing")
print("Key features:")
print("  ✓ Scheduling: Cron, manual, or event-triggered")
print("  ✓ Orchestration: Multiple tasks with dependencies")
print("  ✓ Monitoring: Email alerts, Slack notifications")
print("  ✓ Retries: Auto-retry on failures")
print("  ✓ Concurrency: Control parallel runs\n")

print("="*70)
print("JOB vs DLT PIPELINE")
print("="*70)

print("\nDatabricks Job:")
print("  • General-purpose workflow orchestration")
print("  • Run notebooks, Python files, JARs, SQL")
print("  • Manual dependency definition")
print("  • Flexible scheduling (cron, triggers)")
print("  • Use for: ETL, ML training, reporting\n")

print("Delta Live Tables:")
print("  • Specialized for data pipelines")
print("  • Declarative table definitions")
print("  • Auto dependency inference")
print("  • Built-in data quality")
print("  • Use for: Bronze-Silver-Gold architectures\n")

print("="*70)
print("JOB TASK TYPES")
print("="*70)

print("\n1. NOTEBOOK Task")
print("   • Run a Databricks notebook")
print("   • Pass parameters")
print("   • Most common type\n")

print("2. PYTHON Script Task")
print("   • Run a Python file (.py)")
print("   • For production-grade code")
print("   • Version controlled\n")

print("3. JAR Task")
print("   • Run Scala/Java application")
print("   • For compiled Spark jobs\n")

print("4. SPARK Submit Task")
print("   • Generic Spark application")
print("   • Maximum control\n")

print("5. PIPELINE Task")
print("   • Run a DLT pipeline")
print("   • Trigger pipeline refresh\n")

print("6. SQL Task")
print("   • Run SQL queries")
print("   • Against SQL Warehouse\n")

print("="*70)
print("JOB SCHEDULING OPTIONS")
print("="*70)

print("\n1. MANUAL")
print("   • Run on-demand")
print("   • Click 'Run Now'\n")

print("2. SCHEDULED (Cron)")
print("   • Daily at 2 AM: 0 0 2 * * ?")
print("   • Every hour: 0 0 * * * ?")
print("   • Every Monday: 0 0 8 * * MON\n")

print("3. FILE ARRIVAL")
print("   • Trigger when new files land")
print("   • Monitor specific paths\n")

print("4. CONTINUOUS")
print("   • Always running")
print("   • For streaming workloads\n")

print("="*70)
print("TASK DEPENDENCIES")
print("="*70)

print("\nJobs support multi-task workflows:")
print("\nExample: ETL Pipeline")
print("  Task 1: Ingest raw data (Bronze)")
print("  Task 2: Clean data (Silver) - depends on Task 1")
print("  Task 3: Aggregate data (Gold) - depends on Task 2")
print("  Task 4: Send email report - depends on Task 3\n")

print("Tasks run in parallel when no dependencies\n")

print("="*70)
print("CLUSTER CONFIGURATION")
print("="*70)

print("\n1. ALL-PURPOSE Cluster (Shared)")
print("   • Use existing cluster")
print("   • Good for development")
print("   • Higher cost\n")

print("2. JOB Cluster (New for each run)")
print("   • Creates cluster per run")
print("   • Production best practice")
print("   • Lower cost")
print("   • Auto-terminates after job\n")

print("3. SERVERLESS")
print("   • No cluster management")
print("   • Auto-scaling")
print("   • Fastest startup\n")

print("="*70)
print("JOB MONITORING & ALERTS")
print("="*70)

print("\nMonitoring:")
print("  ✓ View run history")
print("  ✓ Task execution times")
print("  ✓ Error logs")
print("  ✓ Lineage visualization\n")

print("Alerts:")
print("  ✓ Email on success/failure")
print("  ✓ Slack notifications")
print("  ✓ Webhook integrations")
print("  ✓ Custom alerts\n")

print("="*70)
print("WHEN TO USE JOBS")
print("="*70)

print("\n✓ Use Jobs when:")
print("  • Need scheduled execution (daily, hourly, etc.)")
print("  • Multiple tasks with dependencies")
print("  • Production data pipelines")
print("  • ML model training/inference")
print("  • Reporting workflows")
print("  • Data quality checks\n")

print("❌ Don't use Jobs when:")
print("  • Interactive analysis (use notebooks directly)")
print("  • Ad-hoc queries (use SQL Editor)")
print("  • One-time tasks\n")

print("="*70)
print("NEXT: See code examples for creating jobs")
print("="*70)

In [0]:
# ============================================
# DELTA LIVE TABLES (DLT) - Overview
# ============================================
# DLT is a declarative framework for building reliable data pipelines

print("="*70)
print("WHAT IS DELTA LIVE TABLES (DLT)?")
print("="*70)

print("\nDLT is a framework for building and managing data pipelines")
print("Key features:")
print("  ✓ Declarative: Define WHAT you want, not HOW to do it")
print("  ✓ Auto-scaling: Automatically scales compute")
print("  ✓ Data quality: Built-in expectations for validation")
print("  ✓ Monitoring: Automatic lineage and quality metrics")
print("  ✓ Reliability: Auto-retry on failures\n")

print("="*70)
print("DLT vs Traditional ETL")
print("="*70)

print("\nTraditional ETL:")
print("  ❌ Manual orchestration (schedule jobs)")
print("  ❌ Manual error handling")
print("  ❌ Manual quality checks")
print("  ❌ Manual dependency management")

print("\nDelta Live Tables:")
print("  ✓ Auto orchestration (dependencies inferred)")
print("  ✓ Auto retry and recovery")
print("  ✓ Built-in expectations (quality rules)")
print("  ✓ Auto dependency graph\n")

print("="*70)
print("DLT ARCHITECTURE")
print("="*70)

print("\nBronze Layer (Raw):")
print("  • Ingest raw data from sources")
print("  • Minimal transformation")
print("  • Keep original format\n")

print("Silver Layer (Cleaned):")
print("  • Clean and validate data")
print("  • Apply business logic")
print("  • Enforce data quality\n")

print("Gold Layer (Aggregated):")
print("  • Business-level aggregations")
print("  • Ready for analytics")
print("  • Optimized for queries\n")

print("="*70)
print("DLT TABLE TYPES")
print("="*70)

print("\n1. STREAMING TABLE")
print("   • Processes data incrementally")
print("   • For continuous data sources")
print("   • Exactly-once semantics")
print("   Syntax: @dlt.table() / CREATE STREAMING TABLE\n")

print("2. MATERIALIZED VIEW")
print("   • Always up-to-date")
print("   • Recomputes on upstream changes")
print("   • For batch processing")
print("   Syntax: @dlt.view() / CREATE LIVE TABLE\n")

print("3. VIEW")
print("   • Not materialized")
print("   • Computed on-demand")
print("   • Intermediate transformations")
print("   Syntax: @dlt.view() / CREATE TEMPORARY LIVE VIEW\n")

print("="*70)
print("DATA QUALITY - EXPECTATIONS")
print("="*70)

print("\nExpectations validate data quality:")
print("\n1. WARN: Log violations but keep data")
print("   @dlt.expect('valid_id', 'id IS NOT NULL')\n")

print("2. DROP: Remove invalid records")
print("   @dlt.expect_or_drop('valid_age', 'age > 0')\n")

print("3. FAIL: Stop pipeline on violations")
print("   @dlt.expect_or_fail('no_nulls', 'city IS NOT NULL')\n")

print("="*70)
print("WHEN TO USE DLT")
print("="*70)

print("\n✓ Use DLT when:")
print("  • Building multi-stage pipelines (Bronze → Silver → Gold)")
print("  • Need automatic dependency management")
print("  • Require built-in data quality checks")
print("  • Want automatic lineage tracking")
print("  • Processing streaming or batch data\n")

print("❌ Don't use DLT when:")
print("  • Simple one-time data loads")
print("  • Ad-hoc exploratory analysis")
print("  • Complex custom orchestration needed\n")

print("="*70)
print("NEXT: See code examples in following cells")
print("="*70)

# 📋 TASK ORCHESTRATION - Complete Guide

## Navigation: Follow This Order

The Task Orchestration content is organized in 7 steps. Click the links below to jump to each section:

---

### **📖 Overview**
**[Task Orchestration - Overview](#cell-c7486492-c3bf-49cc-9401-7ec8580044eb)**  
Introduction to task orchestration concepts and patterns

---

### **Step-by-Step Orchestration Patterns**

**[Step 1: Sequential Orchestration](#cell-275c2251-52f0-4c70-b9e1-f05a1351ae78)**  
`Task A → Task B → Task C → Task D`  
Basic linear workflow where each task waits for the previous one

**[Step 2: Parallel Orchestration (Fan-Out)](#cell-f8f85da4-8d87-415b-9ab9-4be4975d97a3)**  
```
Task A ─┬→ Task B
        ├→ Task C  
        └→ Task D
```
Multiple tasks run simultaneously after a trigger task

**[Step 3: Fan-In Orchestration (Merge)](#cell-d8b32a82-5515-44e6-a2ae-00e120c40d02)**  
```
Task A ─┐
Task B ─┤→ Task D
Task C ─┘
```
Wait for multiple parallel tasks before proceeding

**[Step 4: Conditional Orchestration (If-Then-Else)](#cell-6d497311-379f-4c23-9fe4-315c20a501d0)**  
`Task A → Decision → Task B (if true) or Task C (if false)`  
Dynamic routing based on conditions

**[Step 5: Diamond Orchestration (Parallel + Merge)](#cell-0a69ca18-8d8c-4add-bfa4-3e3352fdb571)**  
```
         ─┬→ Task B ─┐
Task A ──┤           ├→ Task D
         └→ Task C ─┘
```
Combines fan-out and fan-in patterns

---

### **🎯 Real-World Example**

**[Complete E-commerce Pipeline](#cell-f2ae7aab-277f-4231-9279-702e515d3cfe)**  
Full production-ready example combining all patterns

---

## Quick Summary

| Pattern | When to Use | Example |
|---------|-------------|----------|
| **Sequential** | Simple ETL, data quality checks | Extract → Transform → Load |
| **Parallel (Fan-Out)** | Independent processing | Ingest from multiple sources |
| **Fan-In (Merge)** | Combine results | Join features for ML |
| **Conditional** | Dynamic workflows | Data quality gates |
| **Diamond** | Complex pipelines | ML feature engineering |

---

**💡 Tip**: Start with Step 1 (Sequential) and progress through each pattern. Each builds on the previous concepts.

**📝 Note**: All code examples use Databricks Jobs API and are ready to run.

In [0]:
# ============================================
# DELTA LIVE TABLES - Python Pipeline Example
# ============================================
# This code would be in a DLT notebook/pipeline
# Note: This is for demonstration - requires DLT pipeline to run

print("="*70)
print("DLT PYTHON CODE STRUCTURE")
print("="*70)

print("\n# Import DLT library")
print("import dlt")
print("from pyspark.sql.functions import *\n")

print("="*70)
print("BRONZE LAYER - Raw Data Ingestion")
print("="*70)

print("""
# Bronze: Ingest raw JSON files using Auto Loader
@dlt.table(
    name="bronze_customers",
    comment="Raw customer data from source",
    table_properties={"quality": "bronze"}
)
def bronze_customers():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.inferColumnTypes", "true")
            .load("/Volumes/main/default/raw_data/customers/")
            .withColumn("ingestion_time", current_timestamp())
    )
""")

print("\n" + "="*70)
print("SILVER LAYER - Cleaned and Validated")
print("="*70)

print("""
# Silver: Clean data with quality checks
@dlt.table(
    name="silver_customers",
    comment="Cleaned customer data with quality checks",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_id", "customer_id IS NOT NULL")
@dlt.expect_or_drop("valid_email", "email LIKE '%@%'")
@dlt.expect("recent_data", "ingestion_time > current_date() - INTERVAL 30 DAYS")
def silver_customers():
    return (
        dlt.read_stream("bronze_customers")
            .select(
                col("customer_id").cast("integer"),
                col("name"),
                col("email"),
                col("city"),
                col("signup_date").cast("date"),
                col("ingestion_time")
            )
            .dropDuplicates(["customer_id"])
    )
""")

print("\n" + "="*70)
print("GOLD LAYER - Business Aggregations")
print("="*70)

print("""
# Gold: Business-level aggregations
@dlt.table(
    name="gold_customers_by_city",
    comment="Customer counts by city - analytics ready",
    table_properties={"quality": "gold"}
)
def gold_customers_by_city():
    return (
        dlt.read("silver_customers")
            .groupBy("city")
            .agg(
                count("customer_id").alias("customer_count"),
                min("signup_date").alias("first_signup"),
                max("signup_date").alias("last_signup")
            )
    )
""")

print("\n" + "="*70)
print("STREAMING TABLE vs MATERIALIZED VIEW")
print("="*70)

print("""
# STREAMING TABLE: For continuous processing
@dlt.table(
    name="streaming_orders"
)
def streaming_orders():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .load("/Volumes/main/default/raw_data/orders/")
    )

# MATERIALIZED VIEW: For batch processing
@dlt.table(
    name="daily_sales"
)
def daily_sales():
    return (
        dlt.read("streaming_orders")
            .groupBy(to_date("order_timestamp").alias("date"))
            .agg(sum("amount").alias("total_sales"))
    )

# VIEW (not materialized): Intermediate transformation
@dlt.view(
    name="filtered_orders"
)
def filtered_orders():
    return dlt.read("streaming_orders").filter(col("status") == "completed")
""")

print("\n" + "="*70)
print("DATA QUALITY EXPECTATIONS")
print("="*70)

print("""
# Example: Multiple expectations on one table
@dlt.table(name="validated_orders")
@dlt.expect_or_fail("valid_order_id", "order_id IS NOT NULL")
@dlt.expect_or_drop("positive_amount", "amount > 0")
@dlt.expect("recent_orders", "order_date >= current_date() - 90")
def validated_orders():
    return dlt.read_stream("streaming_orders")

# Expectation Actions:
# - expect:          WARN - log violations, keep all data
# - expect_or_drop:  DROP - remove invalid records
# - expect_or_fail:  FAIL - stop pipeline if violated
""")

print("\n" + "="*70)
print("COMPLETE EXAMPLE: Full Pipeline")
print("="*70)

print("""
import dlt
from pyspark.sql.functions import *

# Bronze: Raw ingestion
@dlt.table(name="bronze_sales")
def bronze_sales():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .load("/Volumes/main/default/raw_data/sales/")
    )

# Silver: Cleaned
@dlt.table(name="silver_sales")
@dlt.expect_or_drop("valid_sale_id", "sale_id IS NOT NULL")
@dlt.expect_or_drop("positive_amount", "amount > 0")
def silver_sales():
    return (
        dlt.read_stream("bronze_sales")
            .select(
                col("sale_id").cast("integer"),
                col("customer_id").cast("integer"),
                col("amount").cast("decimal(10,2)"),
                col("sale_date").cast("date")
            )
    )

# Gold: Analytics
@dlt.table(name="gold_monthly_revenue")
def gold_monthly_revenue():
    return (
        dlt.read("silver_sales")
            .withColumn("month", date_format("sale_date", "yyyy-MM"))
            .groupBy("month")
            .agg(sum("amount").alias("total_revenue"))
    )
""")

print("\n" + "="*70)
print("HOW TO CREATE A DLT PIPELINE")
print("="*70)

print("\n1. Create a notebook with DLT code (like above)")
print("2. Go to Workflows > Delta Live Tables")
print("3. Click 'Create Pipeline'")
print("4. Configure:")
print("   - Pipeline name")
print("   - Notebook path")
print("   - Target schema (database)")
print("   - Cluster configuration")
print("5. Click 'Create' and 'Start'\n")

print("Pipeline will:")
print("  ✓ Parse your code")
print("  ✓ Build dependency graph")
print("  ✓ Execute tables in order")
print("  ✓ Monitor data quality")
print("  ✓ Provide lineage visualization\n")

print("="*70)
print("KEY CONCEPTS SUMMARY")
print("="*70)

print("""
@dlt.table()       - Creates a materialized table
@dlt.view()        - Creates a non-materialized view
dlt.read()         - Read from another DLT table (batch)
dlt.read_stream()  - Read from another DLT table (streaming)

@dlt.expect()           - WARN on violations
@dlt.expect_or_drop()   - DROP invalid records
@dlt.expect_or_fail()   - FAIL pipeline on violations
""")

In [0]:
# ============================================
# DELTA LIVE TABLES - SQL Syntax
# ============================================
# DLT also supports SQL syntax for pipeline definition

print("="*70)
print("DLT SQL CODE EXAMPLES")
print("="*70)

print("\n--- BRONZE LAYER ---\n")
print("""
CREATE OR REFRESH STREAMING TABLE bronze_customers
COMMENT "Raw customer data from JSON files"
TBLPROPERTIES ("quality" = "bronze")
AS SELECT 
  *,
  current_timestamp() as ingestion_time
FROM cloud_files(
  "/Volumes/main/default/raw_data/customers/",
  "json",
  map("cloudFiles.inferColumnTypes", "true")
);
""")

print("\n--- SILVER LAYER with EXPECTATIONS ---\n")
print("""
CREATE OR REFRESH STREAMING TABLE silver_customers (
  CONSTRAINT valid_id EXPECT (customer_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_email EXPECT (email LIKE '%@%') ON VIOLATION DROP ROW,
  CONSTRAINT recent_data EXPECT (ingestion_time > current_date() - INTERVAL 30 DAYS)
)
COMMENT "Cleaned customer data with quality checks"
TBLPROPERTIES ("quality" = "silver")
AS SELECT
  CAST(customer_id AS INTEGER) as customer_id,
  name,
  email,
  city,
  CAST(signup_date AS DATE) as signup_date,
  ingestion_time
FROM STREAM(bronze_customers);
""")

print("\n--- GOLD LAYER - Aggregations ---\n")
print("""
CREATE OR REFRESH LIVE TABLE gold_customers_by_city
COMMENT "Customer counts by city - analytics ready"
TBLPROPERTIES ("quality" = "gold")
AS SELECT
  city,
  COUNT(customer_id) as customer_count,
  MIN(signup_date) as first_signup,
  MAX(signup_date) as last_signup
FROM LIVE.silver_customers
GROUP BY city;
""")

print("\n--- VIEW (Intermediate) ---\n")
print("""
CREATE OR REFRESH TEMPORARY LIVE VIEW active_customers
AS SELECT *
FROM LIVE.silver_customers
WHERE status = 'active';
""")

print("\n" + "="*70)
print("SQL SYNTAX KEY POINTS")
print("="*70)

print("""
Streaming Table:
  CREATE OR REFRESH STREAMING TABLE table_name ...
  FROM STREAM(source_table)

Materialized View:
  CREATE OR REFRESH LIVE TABLE table_name ...
  FROM LIVE.source_table

View (Not Materialized):
  CREATE OR REFRESH TEMPORARY LIVE VIEW view_name ...

Expectations:
  CONSTRAINT name EXPECT (condition) ON VIOLATION [DROP ROW | FAIL UPDATE]

Auto Loader:
  FROM cloud_files(path, format, options)
""")

print("\n" + "="*70)
print("END OF DLT CODE EXAMPLES")
print("="*70)

In [0]:
# ============================================
# CHANGE DATA FEED (CDF) - Overview
# ============================================
# CDF tracks row-level changes in Delta tables

print("="*70)
print("WHAT IS CHANGE DATA FEED (CDF)?")
print("="*70)

print("\nChange Data Feed captures row-level changes to Delta tables")
print("Tracks: INSERTS, UPDATES, DELETES")
print("\nKey features:")
print("  ✓ Row-level change tracking")
print("  ✓ Query changes between versions")
print("  ✓ Incremental data processing")
print("  ✓ Change Data Capture (CDC) patterns")
print("  ✓ Audit and compliance\n")

print("="*70)
print("HOW CDF WORKS")
print("="*70)

print("\nCDF adds metadata to track changes:")
print("  • _change_type: insert, update_preimage, update_postimage, delete")
print("  • _commit_version: Version when change occurred")
print("  • _commit_timestamp: Timestamp of change\n")

print("Change Types:")
print("  insert            - New row added")
print("  update_preimage   - Row BEFORE update")
print("  update_postimage  - Row AFTER update")
print("  delete            - Row deleted\n")

print("="*70)
print("ENABLING CDF")
print("="*70)

print("\nOption 1: Enable on existing table")
print("  ALTER TABLE table_name SET TBLPROPERTIES (delta.enableChangeDataFeed = true)\n")

print("Option 2: Enable during table creation")
print("  CREATE TABLE table_name (...) TBLPROPERTIES (delta.enableChangeDataFeed = true)\n")

print("Option 3: Enable for entire session")
print("  spark.conf.set('spark.databricks.delta.properties.defaults.enableChangeDataFeed', 'true')\n")

print("="*70)
print("QUERYING CHANGES")
print("="*70)

print("\nSQL Syntax:")
print("  SELECT * FROM table_changes('table_name', start_version, end_version)")
print("  SELECT * FROM table_changes('table_name', start_timestamp, end_timestamp)\n")

print("Python API:")
print("  spark.read.format('delta')")
print("       .option('readChangeFeed', 'true')")
print("       .option('startingVersion', 0)")
print("       .table('table_name')\n")

print("="*70)
print("USE CASES")
print("="*70)

print("\n✓ Incremental ETL:")
print("  Process only changed data since last run")
print("  Reduce processing time and cost\n")

print("✓ Slowly Changing Dimensions (SCD Type 2):")
print("  Track historical changes in dimension tables")
print("  Maintain full history\n")

print("✓ Audit Logs:")
print("  Track who changed what and when")
print("  Compliance and governance\n")

print("✓ Data Replication:")
print("  Sync changes to downstream systems")
print("  Real-time data pipelines\n")

print("✓ Materialized Views:")
print("  Incrementally update aggregations")
print("  Avoid full recomputation\n")

print("="*70)
print("NEXT: See working code examples")  
print("="*70)

In [0]:
# ============================================
# CREATE TABLE WITH CDF ENABLED
# ============================================

from pyspark.sql.functions import *

# Table name
table_name = "main.default.cdf_demo_products"

print(f"Creating table: {table_name}")
print("="*70)

# Drop table if exists (for demo purposes)
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# Create initial data
data = [
    (1, "Laptop", 1200.00, "Electronics", "2026-01-15"),
    (2, "Mouse", 25.00, "Electronics", "2026-01-16"),
    (3, "Keyboard", 75.00, "Electronics", "2026-01-17"),
    (4, "Monitor", 350.00, "Electronics", "2026-01-18"),
    (5, "Desk Chair", 250.00, "Furniture", "2026-01-19")
]

df = spark.createDataFrame(data, ["product_id", "product_name", "price", "category", "added_date"])

# Write with CDF enabled
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(table_name)

print(f"✓ Table created with CDF enabled")
print(f"✓ Initial data: {df.count()} rows\n")

# Verify CDF is enabled
props = spark.sql(f"SHOW TBLPROPERTIES {table_name}").filter("key = 'delta.enableChangeDataFeed'").collect()
if props:
    print(f"CDF Status: {props[0]['value']}\n")

# Show initial data
print("Initial data:")
display(spark.table(table_name))

# Check version
history = spark.sql(f"DESCRIBE HISTORY {table_name}").select("version", "operation").first()
print(f"\nCurrent version: {history['version']}")
print(f"Last operation: {history['operation']}")

In [0]:
# ============================================
# PERFORM CHANGES: INSERT, UPDATE, DELETE
# ============================================

from delta.tables import DeltaTable

table_name = "main.default.cdf_demo_products"

print("="*70)
print("PERFORMING CHANGES TO TRACK")
print("="*70)

# Get initial version
initial_version = spark.sql(f"DESCRIBE HISTORY {table_name}").select("version").first()[0]
print(f"\nStarting version: {initial_version}\n")

# CHANGE 1: INSERT new products
print("CHANGE 1: INSERT new products")
print("-" * 40)
new_products = [
    (6, "USB Cable", 15.00, "Electronics", "2026-03-29"),
    (7, "Desk Lamp", 45.00, "Furniture", "2026-03-29")
]

df_new = spark.createDataFrame(new_products, ["product_id", "product_name", "price", "category", "added_date"])
df_new.write.format("delta").mode("append").saveAsTable(table_name)

version_after_insert = spark.sql(f"DESCRIBE HISTORY {table_name}").select("version").first()[0]
print(f"✓ Inserted 2 rows (Version {version_after_insert})\n")

# CHANGE 2: UPDATE prices (price increase)
print("CHANGE 2: UPDATE prices for Electronics")
print("-" * 40)

delta_table = DeltaTable.forName(spark, table_name)
delta_table.update(
    condition="category = 'Electronics'",
    set={"price": "price * 1.10"}  # 10% price increase
)

version_after_update = spark.sql(f"DESCRIBE HISTORY {table_name}").select("version").first()[0]
print(f"✓ Updated Electronics prices +10% (Version {version_after_update})\n")

# CHANGE 3: DELETE a product
print("CHANGE 3: DELETE discontinued product")
print("-" * 40)

delta_table.delete("product_id = 2")  # Delete Mouse

version_after_delete = spark.sql(f"DESCRIBE HISTORY {table_name}").select("version").first()[0]
print(f"✓ Deleted product_id=2 (Version {version_after_delete})\n")

# Show final state
print("="*70)
print("CURRENT TABLE STATE")
print("="*70)
final_df = spark.table(table_name)
print(f"Total rows: {final_df.count()}\n")
display(final_df.orderBy("product_id"))

# Summary
print("\n" + "="*70)
print("CHANGE SUMMARY")
print("="*70)
print(f"Version {initial_version} → {version_after_insert}: INSERT 2 rows")
print(f"Version {version_after_insert} → {version_after_update}: UPDATE Electronics prices")
print(f"Version {version_after_update} → {version_after_delete}: DELETE product_id=2")
print(f"\nWe can now query these changes using CDF!")

In [0]:
%sql
-- ============================================
-- QUERY CHANGES USING table_changes() - SQL
-- ============================================
-- Query all changes since version 0

SELECT 
  product_id,
  product_name,
  price,
  category,
  _change_type,
  _commit_version,
  _commit_timestamp
FROM table_changes('main.default.cdf_demo_products', 0)
ORDER BY _commit_version, product_id, _change_type;

In [0]:
# ============================================
# QUERY CHANGES USING PYTHON API
# ============================================

table_name = "main.default.cdf_demo_products"

print("="*70)
print("QUERYING CHANGES WITH PYTHON API")
print("="*70)

# Method 1: Query from specific version
print("\nMethod 1: Query from version 0 to latest")
print("-" * 40)

changes_df = spark.read \
    .format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 0) \
    .table(table_name)

print(f"Total changes: {changes_df.count()}\n")

# Show all changes
print("All changes:")
display(changes_df.select(
    "product_id", 
    "product_name", 
    "price", 
    "_change_type", 
    "_commit_version"
).orderBy("_commit_version", "product_id", "_change_type"))

# Method 2: Query between versions
print("\n" + "="*70)
print("Method 2: Query specific version range")
print("-" * 40)

# Get version range
versions = spark.sql(f"DESCRIBE HISTORY {table_name}").select("version").collect()
max_version = max([v[0] for v in versions])

if max_version >= 2:
    changes_range = spark.read \
        .format("delta") \
        .option("readChangeFeed", "true") \
        .option("startingVersion", 1) \
        .option("endingVersion", 2) \
        .table(table_name)
    
    print(f"Changes between version 1 and 2: {changes_range.count()} rows")
    display(changes_range.select("product_id", "product_name", "_change_type", "_commit_version"))
else:
    print("Not enough versions for range query")

# Method 3: Query by timestamp
print("\n" + "="*70)
print("Method 3: Query by timestamp (last hour)")
print("-" * 40)

from datetime import datetime, timedelta

start_time = (datetime.now() - timedelta(hours=1)).strftime("%Y-%m-%d %H:%M:%S")

changes_time = spark.read \
    .format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingTimestamp", start_time) \
    .table(table_name)

print(f"Changes in last hour: {changes_time.count()} rows")

In [0]:
# ============================================
# ANALYZE DIFFERENT CHANGE TYPES
# ============================================

from pyspark.sql.functions import col, round as spark_round

table_name = "main.default.cdf_demo_products"

print("="*70)
print("ANALYZING CHANGE TYPES")
print("="*70)

# Get all changes
changes_df = spark.read \
    .format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 0) \
    .table(table_name)

# Count by change type
print("\nChange type summary:")
change_summary = changes_df.groupBy(col("_change_type")).count().orderBy(col("_change_type"))
display(change_summary)

# Filter INSERTS only
print("\n" + "="*70)
print("INSERTS ONLY")
print("-" * 40)
inserts = changes_df.filter(col("_change_type") == "insert")
print(f"Total inserts: {inserts.count()}\n")
display(inserts.select("product_id", "product_name", "price", col("_commit_version")))

# Filter UPDATES (pre and post images)
print("\n" + "="*70)
print("UPDATES - Before and After")
print("-" * 40)
updates = changes_df.filter(col("_change_type").like("update%")).orderBy("product_id", col("_change_type"))
print(f"Total update records: {updates.count()} (preimage + postimage)\n")
display(updates.select("product_id", "product_name", "price", col("_change_type"), col("_commit_version")))

print("\nNote: Each UPDATE generates 2 records:")
print("  - update_preimage: Row BEFORE the change")
print("  - update_postimage: Row AFTER the change")

# Filter DELETES only
print("\n" + "="*70)
print("DELETES ONLY")
print("-" * 40)
deletes = changes_df.filter(col("_change_type") == "delete")
print(f"Total deletes: {deletes.count()}\n")
if deletes.count() > 0:
    display(deletes.select("product_id", "product_name", "price", col("_commit_version")))

# Compare UPDATE before/after
print("\n" + "="*70)
print("UPDATE COMPARISON - Price Changes")
print("-" * 40)

pre = updates.filter(col("_change_type") == "update_preimage").select(
    col("product_id"),
    col("product_name"),
    col("price").alias("old_price")
)

post = updates.filter(col("_change_type") == "update_postimage").select(
    col("product_id"),
    col("price").alias("new_price")
)

if pre.count() > 0 and post.count() > 0:
    comparison = pre.join(post, "product_id") \
        .withColumn("price_change", spark_round(col("new_price") - col("old_price"), 2)) \
        .withColumn("percent_change", spark_round((col("new_price") - col("old_price")) / col("old_price") * 100, 1))
    
    print("Price changes:")
    display(comparison)
else:
    print("No price updates to compare")

In [0]:
# ============================================
# INCREMENTAL PROCESSING WITH CDF
# ============================================
# Common pattern: Process only new changes since last run

table_name = "main.default.cdf_demo_products"
target_table = "main.default.cdf_processed_changes"

print("="*70)
print("INCREMENTAL PROCESSING PATTERN")
print("="*70)

print("\nUse case: Process only changes since last checkpoint")
print("This is much more efficient than reprocessing entire table\n")

# Step 1: Track last processed version (in production, store this in a control table)
print("Step 1: Get last processed version")
print("-" * 40)

# For demo, let's say we last processed version 0
last_processed_version = 0
print(f"Last processed version: {last_processed_version}\n")

# Step 2: Read only NEW changes since last processed version
print("Step 2: Read only NEW changes")
print("-" * 40)

new_changes = spark.read \
    .format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", last_processed_version + 1) \
    .table(table_name)

if new_changes.count() > 0:
    print(f"New changes found: {new_changes.count()} rows\n")
    
    # Step 3: Process changes (example: filter and transform)
    print("Step 3: Process new changes")
    print("-" * 40)
    
    from pyspark.sql.functions import current_timestamp
    
    processed = new_changes.select(
        "product_id",
        "product_name",
        "price",
        "_change_type",
        "_commit_version",
        "_commit_timestamp"
    ).withColumn("processed_at", current_timestamp())
    
    print("Processed changes:")
    display(processed)
    
    # Step 4: Write to target table
    print("\nStep 4: Write to target table")
    print("-" * 40)
    
    processed.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(target_table)
    
    print(f"✓ Written {processed.count()} changes to {target_table}")
    
    # Step 5: Update checkpoint (in production, update control table)
    current_version = spark.sql(f"DESCRIBE HISTORY {table_name}").select("version").first()[0]
    print(f"\nStep 5: Update checkpoint to version {current_version}")
    print(f"✓ Next run will start from version {current_version + 1}")
    
else:
    print("No new changes to process")

print("\n" + "="*70)
print("INCREMENTAL PATTERN BENEFITS")
print("="*70)
print("✓ Process only changed data (not full table scan)")
print("✓ Lower compute costs")
print("✓ Faster processing")
print("✓ Supports real-time/near-real-time pipelines")
print("✓ Exactly-once processing semantics")

In [0]:
# ============================================
# CDF REAL-WORLD USE CASES
# ============================================

print("="*70)
print("CDF USE CASE 1: AUDIT LOG")
print("="*70)

print("\nTrack all changes for compliance and auditing")
print("\nExample query:")
print("""
SELECT 
  _commit_timestamp as change_time,
  _change_type as operation,
  product_id,
  product_name,
  price
FROM table_changes('products', '2026-01-01', '2026-12-31')
WHERE _change_type IN ('update_postimage', 'delete')
ORDER BY _commit_timestamp DESC;
""")

print("Use for: GDPR compliance, SOX auditing, security monitoring\n")

print("="*70)
print("CDF USE CASE 2: SLOWLY CHANGING DIMENSION (SCD TYPE 2)")
print("="*70)

print("\nMaintain full history of dimension changes")
print("\nExample pattern:")
print("""
# Read changes from source
changes = spark.read \
    .format('delta') \
    .option('readChangeFeed', 'true') \
    .option('startingVersion', last_version) \
    .table('customer_source')

# For each change, create new dimension row with:
# - effective_date = change timestamp
# - end_date = NULL (current record)
# - is_current = True
# - version = version + 1

# Update previous record:
# - end_date = change timestamp  
# - is_current = False
""")

print("Result: Full history of dimension changes over time\n")

print("="*70)
print("CDF USE CASE 3: DOWNSTREAM SYSTEM SYNC")
print("="*70)

print("\nReplicate changes to downstream databases/warehouses")
print("\nExample workflow:")
print("""
1. Read changes since last sync
changes = spark.read \
    .format('delta') \
    .option('readChangeFeed', 'true') \
    .option('startingVersion', last_synced_version) \
    .table('source_table')

2. Apply changes to target system
for row in changes.collect():
    if row._change_type == 'insert':
        target_db.insert(row)
    elif row._change_type == 'update_postimage':
        target_db.update(row)
    elif row._change_type == 'delete':
        target_db.delete(row)

3. Update sync checkpoint
""")

print("Use for: Syncing to Snowflake, Postgres, Elasticsearch, etc.\n")

print("="*70)
print("CDF USE CASE 4: MATERIALIZED VIEW REFRESH")
print("="*70)

print("\nIncrementally update aggregations without full recomputation")
print("\nExample:")
print("""
# Instead of recomputing entire aggregation:
SELECT category, SUM(price) FROM products GROUP BY category

# Process only changes:
changes = read_cdf('products', last_version)

# For inserts: add to aggregation
# For updates: subtract old, add new  
# For deletes: subtract from aggregation

Result: 100x faster for large tables!
""")

print("="*70)
print("CDF USE CASE 5: DATA QUALITY MONITORING")
print("="*70)

print("\nMonitor data quality over time")
print("\nExample:")
print("""
SELECT 
  date_trunc('hour', _commit_timestamp) as hour,
  _change_type,
  COUNT(*) as change_count,
  COUNT(CASE WHEN price < 0 THEN 1 END) as invalid_price_count,
  COUNT(CASE WHEN product_name IS NULL THEN 1 END) as null_name_count
FROM table_changes('products', start_date, end_date)
GROUP BY hour, _change_type
ORDER BY hour DESC;
""")

print("Alert on anomalies: sudden spike in deletes, invalid data patterns\n")

print("="*70)
print("CDF BEST PRACTICES")
print("="*70)

print("""
✓ Enable CDF at table creation (not retroactive)
✓ Store checkpoint/watermark in control table
✓ Use version-based checkpoints (more reliable than timestamp)
✓ Handle update_preimage and update_postimage correctly
✓ Combine with VACUUM retention for storage management
✓ Test incremental logic thoroughly
✓ Monitor CDF query performance
✓ Consider CDF overhead for high-volume tables
""")

print("\n" + "="*70)
print("END OF CHANGE DATA FEED TUTORIAL")
print("="*70)

In [0]:
# ============================================
# CHANGE DATA FEED - QUICK REFERENCE
# ============================================

print("="*70)
print("CDF QUICK REFERENCE GUIDE")
print("="*70)

print("\n" + "="*70)
print("1. ENABLE CDF")
print("="*70)

print("""
-- On existing table
ALTER TABLE table_name 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- During table creation
CREATE TABLE table_name (...) 
TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- For session (all new tables)
SPARK.conf.set('spark.databricks.delta.properties.defaults.enableChangeDataFeed', 'true')
""")

print("="*70)
print("2. QUERY CHANGES - SQL")
print("="*70)

print("""
-- By version range
SELECT * FROM table_changes('table_name', 0, 10);

-- By timestamp range  
SELECT * FROM table_changes('table_name', '2026-01-01', '2026-12-31');

-- Since specific version
SELECT * FROM table_changes('table_name', 5);
""")

print("="*70)
print("3. QUERY CHANGES - PYTHON")
print("="*70)

print("""
# By version
df = spark.read \\
    .format('delta') \\
    .option('readChangeFeed', 'true') \\
    .option('startingVersion', 0) \\
    .option('endingVersion', 10) \\
    .table('table_name')

# By timestamp
df = spark.read \\
    .format('delta') \\
    .option('readChangeFeed', 'true') \\
    .option('startingTimestamp', '2026-01-01') \\
    .option('endingTimestamp', '2026-12-31') \\
    .table('table_name')
""")

print("="*70)
print("4. CDF METADATA COLUMNS")
print("="*70)

print("""
_change_type        : insert | update_preimage | update_postimage | delete
_commit_version     : Delta table version when change occurred
_commit_timestamp   : Timestamp when change was committed
""")

print("="*70)
print("5. CHANGE TYPE MEANINGS")
print("="*70)

print("""
insert              : New row inserted
update_preimage     : Row state BEFORE update (old values)
update_postimage    : Row state AFTER update (new values)
delete              : Row was deleted

Note: Each UPDATE generates TWO records (pre + post)
""")

print("="*70)
print("6. COMMON FILTERS")
print("="*70)

print("""
-- Only inserts
WHERE _change_type = 'insert'

-- Only updates (latest values)
WHERE _change_type = 'update_postimage'

-- Only deletes
WHERE _change_type = 'delete'

-- Inserts and final update values
WHERE _change_type IN ('insert', 'update_postimage')
""")

print("="*70)
print("7. INCREMENTAL PROCESSING TEMPLATE")
print("="*70)

print("""
# Step 1: Get last checkpoint
last_version = get_checkpoint()  # from control table

# Step 2: Read new changes
changes = spark.read \\
    .format('delta') \\
    .option('readChangeFeed', 'true') \\
    .option('startingVersion', last_version + 1) \\
    .table('source_table')

# Step 3: Process changes
processed = transform(changes)

# Step 4: Write results
processed.write.mode('append').saveAsTable('target_table')

# Step 5: Update checkpoint
current_version = get_current_version('source_table')
update_checkpoint(current_version)
""")

print("="*70)
print("8. LIMITATIONS")
print("="*70)

print("""
• CDF is NOT retroactive (only tracks changes after enabling)
• CDF adds small storage overhead (~5-10%)
• VACUUM will delete CDF data beyond retention period
• Default retention: 30 days
• Cannot disable CDF once enabled (table property is immutable)
""")

print("\n" + "="*70)
print("RESOURCES")
print("="*70)
print("Docs: docs.databricks.com/delta/delta-change-data-feed.html")
print("Tutorial complete! Run cells sequentially to see CDF in action.")
print("="*70)

In [0]:
# ============================================
# TIME TRAVEL: Query Historical Data
# ============================================
# Delta Lake keeps ALL versions of your table
# You can query any previous version using VERSION AS OF

table_name = "main.default.user_data_external"

print("=== What is Time Travel? ===")
print("Delta Lake maintains complete history of all changes")
print("You can query table state at any point in time!\n")

# Show all versions
print("=== Table Version History ===")
history = spark.sql(f"DESCRIBE HISTORY {table_name}").select(
    "version", "timestamp", "operation", "operationMetrics"
)

print("Available versions:")
for row in history.collect():
    print(f"  Version {row['version']}: {row['operation']} at {row['timestamp']}")

# Get latest version number
latest_version = history.agg({"version": "max"}).collect()[0][0]
print(f"\nLatest version: {latest_version}\n")

# METHOD 1: Query by VERSION number
print("=== Method 1: Query by VERSION ===")
if latest_version > 0:
    previous_version = latest_version - 1
    print(f"Querying version {previous_version}...")
    
    df_v1 = spark.sql(f"""
        SELECT * FROM {table_name} VERSION AS OF {previous_version}
        LIMIT 5
    """)
    
    print(f"Rows in version {previous_version}: {df_v1.count()}")
    display(df_v1)
else:
    print("Only one version exists")

# METHOD 2: Query by TIMESTAMP
print("\n=== Method 2: Query by TIMESTAMP ===")
print("Query table as it was at a specific date/time")
print("Example: SELECT * FROM table TIMESTAMP AS OF '2026-03-29T00:00:00'\n")

# METHOD 3: Python API
print("=== Method 3: Python API ===")
if latest_version > 0:
    df_python = spark.read \
        .format("delta") \
        .option("versionAsOf", previous_version) \
        .table(table_name)
    
    print(f"Rows using Python API: {df_python.count()}")

print("\n=== Use Cases for Time Travel ===")
print("✓ Audit: See what data looked like yesterday")
print("✓ Rollback: Compare before/after a bad update")
print("✓ Debug: Investigate when data changed")
print("✓ Compliance: Provide historical snapshots")

In [0]:
%sql
-- Verify the external table
SELECT * FROM main.default.user_data_external LIMIT 10;

In [0]:
%sql
-- Step 1: Compact small files into larger ones for better read performance
OPTIMIZE main.default.user_data_external;

-- Step 2: Z-ORDER optimization
-- Z-ORDER co-locates related data in the same files using multi-dimensional clustering
-- Benefits:
--   - Faster queries with WHERE filters on Z-ORDERed columns
--   - Data skipping: reads only relevant files, not entire table
--   - Best for columns frequently used in WHERE clauses

-- Example: If you often query by city and ingestion_time:
--   SELECT * FROM table WHERE city = 'Bangalore' AND ingestion_time > '2026-01-01'
-- Z-ORDER will cluster rows with same city + time range together in files

OPTIMIZE main.default.user_data_external ZORDER BY (city, ingestion_time);

-- Step 3: Collect column statistics for query optimizer
ANALYZE TABLE main.default.user_data_external COMPUTE STATISTICS FOR ALL COLUMNS;

-- When to use Z-ORDER:
-- ✓ High-cardinality columns (many unique values): city, user_id, date
-- ✓ Columns in WHERE clauses of frequent queries
-- ✓ Columns used in JOIN conditions
-- ✗ Low-cardinality columns (few values): status with 2-3 values
-- ✗ Columns you rarely filter on

In [0]:
%sql
-- View table optimization history
-- Shows OPTIMIZE and Z-ORDER operations performed
DESCRIBE HISTORY main.default.user_data_external;

-- View detailed table information including file count
-- After OPTIMIZE: fewer, larger files
-- Before OPTIMIZE: many small files
DESCRIBE DETAIL main.default.user_data_external;

In [0]:
%sql
-- Query filtered by Z-ORDERed columns (city, ingestion_time)
-- After Z-ORDER, this query will:
--   1. Skip files that don't contain 'Bangalore'
--   2. Skip files outside the time range
--   3. Read only relevant files (data skipping)

SELECT 
    city,
    COUNT(*) as user_count,
    MIN(ingestion_time) as first_ingestion,
    MAX(ingestion_time) as last_ingestion
FROM main.default.user_data_external
WHERE city = 'Bangalore'
  AND ingestion_time >= '2026-03-01'
GROUP BY city;

-- Check optimization stats:
-- DESCRIBE HISTORY main.default.user_data_external;

In [0]:
# View Delta Table Optimization History Using Python
# Shows what OPTIMIZE and Z-ORDER operations were performed

from delta.tables import DeltaTable
import pandas as pd

table_name = "main.default.user_data_external"

print("=== Delta Table History ===")
history_df = spark.sql(f"DESCRIBE HISTORY {table_name}")

# Show recent operations
recent_ops = history_df.select(
    "version", 
    "timestamp", 
    "operation", 
    "operationParameters"
).limit(10)

print("\nRecent Operations:")
for row in recent_ops.collect():
    print(f"\nVersion: {row['version']}")
    print(f"Time: {row['timestamp']}")
    print(f"Operation: {row['operation']}")
    if row['operation'] == 'OPTIMIZE':
        params = row['operationParameters']
        if 'zOrderBy' in params:
            print(f"Z-ORDER columns: {params['zOrderBy']}")
        print("  → Files compacted and data clustered")

print("\n=== Table Details ===")
detail = spark.sql(f"DESCRIBE DETAIL {table_name}").collect()[0]

print(f"\nTable: {detail['name']}")
print(f"Format: {detail['format']}")
print(f"Location: {detail['location']}")
print(f"Number of files: {detail['numFiles']}")
print(f"Total size: {detail['sizeInBytes']:,} bytes ({detail['sizeInBytes'] / 1024:.2f} KB)")

print("\n=== Column Statistics ===")
stats_df = spark.sql(f"DESCRIBE EXTENDED {table_name}")

# Show table statistics
print("\nTable-level statistics collected for query optimization")
print("These help Spark choose the best query execution plan")

print("\n=== Z-ORDER Summary ===")
print("✓ OPTIMIZE compacts small files → fewer files, faster reads")
print("✓ Z-ORDER clusters data by columns → data skipping enabled")
print("✓ ANALYZE collects statistics → better query plans")
print("\nResult: Faster queries, lower costs, better performance!")

In [0]:
# Demonstrate Z-ORDER Performance Benefits
# After Z-ORDER, queries filtered by (city, ingestion_time) will be MUCH faster

import time
from pyspark.sql.functions import col

table_name = "main.default.user_data_external"

print("=== How Z-ORDER Improves Performance ===")
print("\nWithout Z-ORDER:")
print("  File 1: [Bangalore, Mumbai, Chennai, Pune] - random mix")
print("  File 2: [Bangalore, Mumbai, Chennai, Pune] - random mix")
print("  File 3: [Bangalore, Mumbai, Chennai, Pune] - random mix")
print("  Query: WHERE city = 'Bangalore'")
print("  Result: Must scan ALL 3 files")

print("\nWith Z-ORDER BY (city):")
print("  File 1: [Bangalore, Bangalore, Bangalore] - clustered")
print("  File 2: [Mumbai, Chennai] - clustered")
print("  File 3: [Pune] - clustered")
print("  Query: WHERE city = 'Bangalore'")
print("  Result: Only scans File 1 (data skipping!)\n")

# Query that benefits from Z-ORDER
print("=== Running optimized query ===")
start_time = time.time()

df_result = spark.sql(f"""
    SELECT 
        city,
        COUNT(*) as user_count,
        MIN(ingestion_time) as first_seen,
        MAX(ingestion_time) as last_seen
    FROM {table_name}
    WHERE city = 'Bangalore'
      AND ingestion_time >= '2026-03-01'
    GROUP BY city
""")

result = df_result.collect()
query_time = time.time() - start_time

print(f"Query completed in: {query_time:.4f} seconds")
print(f"Results: {len(result)} rows")
if result:
    print(f"\nData for Bangalore:")
    for row in result:
        print(f"  City: {row['city']}")
        print(f"  User count: {row['user_count']}")
        print(f"  First seen: {row['first_seen']}")
        print(f"  Last seen: {row['last_seen']}")

print("\n=== Key Benefit: Data Skipping ===")
print("Z-ORDER enables 'data skipping' - reading only relevant files")
print("Performance improvement: 10-100x faster for large tables")
print("Cost savings: Read fewer files = lower compute costs")

In [0]:
# Z-ORDER Optimization Using Python/PySpark
# Z-ORDER is a multi-dimensional clustering technique that organizes data
# to improve query performance by co-locating related data in the same files

from delta.tables import DeltaTable

# Reference the Delta table
table_name = "main.default.user_data_external"
delta_table = DeltaTable.forName(spark, table_name)

print("=== Step 1: Check table details BEFORE optimization ===")
detail_before = spark.sql(f"DESCRIBE DETAIL {table_name}").collect()[0]
print(f"Files before: {detail_before['numFiles']}")
print(f"Size before: {detail_before['sizeInBytes']} bytes\n")

# Step 2: Run OPTIMIZE to compact small files
print("=== Step 2: Running OPTIMIZE (compact files) ===")
spark.sql(f"OPTIMIZE {table_name}")
print("✓ Files compacted\n")

# Step 3: Run Z-ORDER by columns frequently used in WHERE clauses
print("=== Step 3: Running Z-ORDER BY (city, ingestion_time) ===")
print("Z-ORDER clusters data by specified columns for faster filtering")
print("Example: WHERE city = 'Bangalore' AND ingestion_time > '2026-01-01'\n")

spark.sql(f"OPTIMIZE {table_name} ZORDER BY (city, ingestion_time)")
print("✓ Data Z-ORDERed\n")

# Step 4: Collect statistics for query optimizer
print("=== Step 4: Collecting column statistics ===")
spark.sql(f"ANALYZE TABLE {table_name} COMPUTE STATISTICS FOR ALL COLUMNS")
print("✓ Statistics collected\n")

# Step 5: Check results AFTER optimization
print("=== Step 5: Table details AFTER optimization ===")
detail_after = spark.sql(f"DESCRIBE DETAIL {table_name}").collect()[0]
print(f"Files after: {detail_after['numFiles']}")
print(f"Size after: {detail_after['sizeInBytes']} bytes")
print(f"\nReduction: {detail_before['numFiles'] - detail_after['numFiles']} fewer files")

print("\n=== Z-ORDER Best Practices ===")
print("✓ Use on high-cardinality columns (city, user_id, date)")
print("✓ Choose columns in frequent WHERE clauses")
print("✓ Order matters: put most selective column first")
print("✗ Avoid low-cardinality columns (status: active/inactive)")
print("✗ Don't Z-ORDER on rarely queried columns")